# CPMK3 Instance Segmentation | Evaluasi

Evaluasi memakai checkpoint dari pelatihan di Kaggle. Tabel menampilkan mask AP50 dan AP50–95 pada set uji asli dan 40 foto HP. Untuk NRP 0, skor pada 320, 640, dan 960 piksel dibandingkan di set validasi untuk ketiga model.

Draf menambah **279** mask, memperbaiki **15** nama kelas dan **1** batas mask, serta mencatat **21** anotasi objek di luar kelas acuan. Jumlah akhirnya **352** mask pada **40** foto HP.

AI membantu menyiapkan draf ini. Pengecekan draf di Roboflow masih diperlukan. File COCO dan catatan revisi ada di Output Kaggle. Jumlah gambar publik per bagian pada ekspor v2 cocok dengan v5. Isi tiap gambar dan label masih perlu dicocokkan.


## 1. Siapkan pustaka

Kaggle sudah menyediakan PyTorch dan torchvision. Sel ini memasang pustaka yang masih dibutuhkan, lalu mengetes NMS untuk mendeteksi masalah versi. Internet dibutuhkan kalau pustaka atau bobot belum tersedia. Ultralytics memakai versi yang sudah ditentukan supaya fungsi yang dipanggil konsisten.

In [ ]:
# Dependensi tambahan; PyTorch/CUDA bawaan Kaggle tidak diganti.
import os, sys, subprocess, importlib.util, importlib.metadata
os.environ.setdefault('YOLO_CONFIG_DIR', '/kaggle/working/yolo-settings' if os.path.isdir('/kaggle/working') else '/tmp/cpmk3-yolo-settings')
os.makedirs(os.environ['YOLO_CONFIG_DIR'], exist_ok=True)
os.environ.setdefault('WANDB_MODE', 'disabled')
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cpmk3-matplotlib')
required = {'ultralytics': 'ultralytics==8.3.228', 'pycocotools': 'pycocotools==2.0.11',
            'yaml': 'pyyaml', 'pandas': 'pandas', 'matplotlib': 'matplotlib',
            'PIL': 'pillow', 'cv2': 'opencv-python-headless', 'scipy': 'scipy', 'IPython': 'ipython'}
install = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
if importlib.util.find_spec('ultralytics') is not None and importlib.metadata.version('ultralytics') != '8.3.228':
    install.append('ultralytics==8.3.228')
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-q', *install])
import copy, csv, gc, hashlib, io, json, math, random, re, shutil, time, warnings, zipfile
from collections import Counter, defaultdict
from contextlib import redirect_stdout
from datetime import datetime, timezone
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from PIL import Image, ImageDraw, ImageFont, ImageOps, ExifTags
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops.misc import FrozenBatchNorm2d
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
from IPython.display import display
# Tangkap konflik torch/torchvision sebelum berjam-jam training.
_ = torchvision.ops.nms(torch.tensor([[0., 0., 2., 2.]]), torch.tensor([1.]), 0.5)
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| ultralytics', importlib.metadata.version('ultralytics'))


## 2. Atur data dan percobaan

Konfigurasi memakai digit NRP 0. Biarkan `DATASET_OVERRIDE=None` supaya notebook mencari data sendiri. Kalau ada beberapa ekspor, isi path yang benar. Isi `HP_LABELS_REVIEWED` setelah pengecekan anotasi HP selesai.

In [ ]:
# ===== BAGIAN YANG BOLEH KAMU UBAH =====
INPUT_ROOT = Path('/kaggle/input')
# Biarkan None untuk deteksi otomatis. Bila ambigu, isi folder ekspor COCO atau ZIP v2 yang tepat.
DATASET_OVERRIDE = None
WORK_ROOT = Path('/kaggle/working/cpmk3_final')
NRP_LAST_DIGIT = 0
RUN_TRAINING = False             # True setelah provenance data latih dikonfirmasi
RUN_EVALUATION = True            # Otomatis menunggu checkpoint jika training belum dijalankan
RUN_NRP_EXPERIMENT = True        # NRP 0: latih YOLO11n pada 320, 640, 960
RUN_FAILURE_ANALYSIS = True
SOURCE_V5_CONFIRMED = False      # True HANYA jika ada bukti asal data latih dari v5 / persetujuan dosen
SOURCE_V5_NOTE = 'Pelatihan memakai ekspor COCO v2 sesuai pilihan pengguna. Jumlah split publik cocok dengan v5 (397/166/359). Verifikasi kesamaan gambar dan label v5 masih terbuka.'              # Tulis URL/bukti asal atau keputusan dosen yang sebenarnya
HP_LABELS_REVIEWED = False       # True setelah review kelengkapan/ketepatan semua mask HP
# Optional: folder input yang berisi file foto asli dengan EXIF dan screenshot proses anotasi.
ORIGINAL_PHOTOS_DIR = None
SCREENSHOTS_DIR = None
# ===== Pengaturan tetap percobaan =====
SEED = 42
BASE_SIZE = 640
RESOLUTIONS = (320, 640, 960)
EPOCHS = 200                     # Batas epoch; deadline waktu bisa menghentikan lebih awal
BATCH_YOLO = 4                   # Sama untuk ketiga ukuran NRP; turunkan SEMUA varian jika OOM
BATCH_MASK = 1
WORKERS = 0                     # Aman untuk notebook; dapat dinaikkan menjadi 2 bila perlu
YOLO_N_MINUTES = 20             # 640: baseline sekaligus varian NRP
YOLO_S_MINUTES = 30
MASK_MINUTES = 40
NRP_MINUTES = 20                # 320 dan 960; total YOLO11n sekitar 60 menit
MAX_SECONDS_PER_MODEL = 7200
PRED_CONF = 0.001               # Ambang rendah untuk kurva precision-recall AP
PRED_MAX_DET = 100
FAIL_SCORE_THR = 0.50
MASK_LR = 0.0025
# Gambar dengan dimensi piksel != metadata COCO dikarantina, bukan diputar/diregangkan dengan tebakan.
QUARANTINE_DIMENSION_MISMATCH = True
# Asal 40 foto HP dari ZIP Foto_40_Terpilih_JPG_Roboflow.zip, BUKAN regex tanggal.
HP_SOURCE_NAMES = set('''20260713_111506.jpg 20260714_105357.jpg 20260715_125526.jpg
20260804_120415.jpg 20260805_114312.jpg 20260812_110307.jpg 20260826_154026.jpg
20260903_073220.jpg 20260903_161001.jpg 20260903_161212.jpg 20261030_143151.jpg
20261118_083036.jpg 20261118_083047.jpg 20261118_083050.jpg 20261118_084035.jpg
20261118_084038.jpg 20261118_084046.jpg 20261118_084140.jpg 20261118_084148.jpg
20261118_084152.jpg 20261118_134539.jpg 20261118_134543.jpg 20261118_134549.jpg
20261118_192611.jpg 20261118_192615.jpg 20261118_192621.jpg 20261118_192648.jpg
20261118_192856.jpg 20261118_192900.jpg 20261118_192904.jpg 20261118_192913.jpg
20261118_192916.jpg 20261118_192932.jpg 20261118_192936.jpg 20261118_192940.jpg
20261118_192944.jpg 20261118_193001.jpg 20261118_193023.jpg 20261118_193029.jpg
20261118_193033.jpg'''.split())
assert len(HP_SOURCE_NAMES) == 40
assert NRP_LAST_DIGIT == 0, 'Notebook ini disesuaikan untuk digit NRP 0 yang kamu konfirmasi.'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
seed_all()
torch.backends.cudnn.benchmark = False
def write_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    tmp.replace(path)
def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()
def source_name(info):
    original = info.get('extra', {}).get('name')
    if original: return Path(original).name
    name = Path(info['file_name']).name
    # Roboflow: source_jpg.rf.HASH.jpg -> source.jpg
    return re.sub(r'_(jpg|jpeg|png)\.rf\.[^.]+\.[^.]+$', lambda m: '.' + m.group(1), name, flags=re.I)
def is_hp(info): return source_name(info) in HP_SOURCE_NAMES
print('Device:', DEVICE, '| input:', INPUT_ROOT, '| output:', WORK_ROOT)


## 3. Cari dan buka dataset

Notebook mencari folder latih, validasi, dan uji dengan JSON COCO. Berkas tambahan dari macOS dilewati. Saat membuka ZIP, notebook memeriksa path dan symlink.

Ekspor v1 memuat foto HP di data latih. Untuk menjaga foto HP sebagai set uji, notebook memilih ekspor yang memisahkannya dari data latih. ZIP yang berisi foto saja dan folder audit dipakai sebagai bukti pendukung.

In [ ]:
def ignored(path):
    return any(part == '__MACOSX' or part.startswith('.') for part in Path(path).parts)
def safe_extract(archive, dest):
    dest = Path(dest).resolve()
    with zipfile.ZipFile(archive) as z:
        infos = [x for x in z.infolist() if not ignored(x.filename)]
        if sum(x.file_size for x in infos) > 8_000_000_000:
            raise ValueError('Arsip melebihi batas ekstraksi 8 GB.')
        for item in z.infolist():
            target = (dest / item.filename).resolve()
            if not target.is_relative_to(dest) or (item.external_attr >> 16) & 0o170000 == 0o120000:
                raise ValueError('Path/symlink tidak aman di ZIP: ' + item.filename)
        for item in infos: z.extract(item, dest)
    return dest

def split_paths(root):
    paths = {}
    for sp, aliases in {'train': ('train',), 'valid': ('valid', 'val', 'validation'), 'test': ('test',)}.items():
        found = [root / a / '_annotations.coco.json' for a in aliases if (root / a / '_annotations.coco.json').is_file()]
        if len(found) != 1: return None
        paths[sp] = found[0]
    return paths

def inspect_candidate(paths):
    data = {sp: json.loads(p.read_text()) for sp, p in paths.items()}
    hp = {sp: sum(is_hp(i) for i in d['images']) for sp, d in data.items()}
    counts = {sp: len(d['images']) for sp, d in data.items()}
    return {'paths': paths, 'data': data, 'counts': counts, 'hp_counts': hp,
            'eligible': hp == {'train': 0, 'valid': 0, 'test': 40}}

# Menampilkan input yang benar-benar dipasang, bukan hardcode slug Kaggle.
if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(f'{INPUT_ROOT} tidak ada. Di Kaggle pilih Add Input; untuk uji lokal ubah INPUT_ROOT.')
for path in sorted(INPUT_ROOT.iterdir()):
    n_images = sum(1 for p in path.rglob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')) if path.is_dir() else 0
    print(path, '| gambar terurai:', n_images)
search_root = Path(DATASET_OVERRIDE) if DATASET_OVERRIDE else INPUT_ROOT
if not search_root.exists(): raise FileNotFoundError(search_root)
candidates = []
if search_root.is_dir():
    roots = sorted({p.parent.parent for p in search_root.rglob('_annotations.coco.json') if not ignored(p)})
    for root in roots:
        paths = split_paths(root)
        if paths: candidates.append(inspect_candidate(paths))
archives = [search_root] if search_root.is_file() else sorted(search_root.rglob('*.zip'))
for archive in archives:
    if ignored(archive): continue
    with zipfile.ZipFile(archive) as z:
        names = [n for n in z.namelist() if n.endswith('/_annotations.coco.json') and not ignored(n)]
        prefixes = {str(Path(n).parent.parent) for n in names}
        for prefix in prefixes:
            def zn(sp): return str(Path(prefix) / sp / '_annotations.coco.json')
            if not all(zn(sp) in names for sp in ('train', 'valid', 'test')): continue
            meta = {sp: json.loads(z.read(zn(sp))) for sp in ('train', 'valid', 'test')}
            hpc = {sp: sum(is_hp(i) for i in d['images']) for sp, d in meta.items()}
            print('ZIP:', archive.name, '| jumlah:', {sp:len(d['images']) for sp,d in meta.items()}, '| HP:', hpc)
            # V1 memiliki HP di train: ditolak sebelum ekstraksi.
            if hpc != {'train': 0, 'valid': 0, 'test': 40}: continue
            extracted = WORK_ROOT / 'extracted' / sha_file(archive)[:16]
            safe_extract(archive, extracted)
            paths = split_paths(extracted / prefix)
            if paths: candidates.append(inspect_candidate(paths))
valid_candidates = [c for c in candidates if c['eligible']]
# Salinan dataset yang identik (mis. ZIP + hasil ekstraknya) tidak membuat pilihan ambigu.
def signature(c):
    return hashlib.sha256(json.dumps(c['data'], sort_keys=True).encode()).hexdigest()
unique = {signature(c): c for c in valid_candidates}
if len(unique) != 1:
    raise RuntimeError(f'Ditemukan {len(unique)} dataset valid. Pasang ZIP v2 final-test40, bukan v1. '
                       'Jika ada beberapa ekspor berbeda, isi DATASET_OVERRIDE dengan satu path tepat.')
CHOSEN = next(iter(unique.values()))
RAW = CHOSEN['data']
RAW_PATHS = CHOSEN['paths']
RAW_DIRS = {sp: p.parent for sp,p in RAW_PATHS.items()}
print('Dipilih:', RAW_PATHS, '\nJumlah sebelum audit:', CHOSEN['counts'])
print('V1, ZIP foto tanpa label, dan folder preview audit tidak dipakai sebagai training tambahan.')


## Cek revisi anotasi 40 foto HP


In [ ]:
patches = sorted(INPUT_ROOT.rglob('hp_annotation_patch.json'))
if len(patches) != 1:
    raise FileNotFoundError(f'Pasang dataset privat revisi anotasi HP. File patch yang ditemukan: {len(patches)}.')
PATCH_FILE = patches[0]
HP_PATCH = json.loads(PATCH_FILE.read_text())
assert HP_PATCH['schema'] == 1
assert HP_PATCH['status'] == 'assisted_draft_review_needed'
test_original_data = RAW['test']
source_digest = hashlib.sha256(json.dumps(test_original_data,sort_keys=True).encode()).hexdigest()
assert source_digest == HP_PATCH['source_test_sha256'], 'Patch hanya cocok dengan ekspor COCO v2 yang digunakan.'
hp_lookup = {i['id']:i for i in test_original_data['images'] if is_hp(i)}
assert len(hp_lookup)==40 and sorted(hp_lookup)==HP_PATCH['source_image_ids']
categories_patch = {c['name']:c['id'] for c in test_original_data['categories']}
assert len(categories_patch)==len(test_original_data['categories'])
changes = {a['ann_id']:a for a in HP_PATCH['edits']}
replacements = {a['ann_id']:a for a in HP_PATCH['outline_replacements']}
assert len(changes)==len(HP_PATCH['edits'])
assert len(replacements)==len(HP_PATCH['outline_replacements'])
before_ids = {a['id'] for a in test_original_data['annotations']}
assert set(changes).issubset(before_ids) and set(replacements).issubset(before_ids)
amended = []
for original in test_original_data['annotations']:
    ann_id, image_id = original['id'], original['image_id']
    if ann_id not in changes and ann_id not in replacements:
        amended.append(original)
        continue
    assert image_id in hp_lookup
    current = copy.deepcopy(original)
    if ann_id in changes:
        row = changes[ann_id]
        assert row['image_id']==image_id and row['photo']==source_name(hp_lookup[image_id])
        assert categories_patch[row['from']] == original['category_id']
        if row['to'] is None:
            assert ann_id not in replacements
            continue
        current['category_id'] = categories_patch[row['to']]
    if ann_id in replacements:
        row = replacements[ann_id]
        assert row['image_id']==image_id
        current.update(row['geometry'])
    amended.append(current)
additions = HP_PATCH['added']
added_ids = {a['id'] for a in additions}
assert len(added_ids)==len(additions) and not added_ids & before_ids
for a in additions:
    assert a['image_id'] in hp_lookup and a['category_id'] in categories_patch.values()
    i = hp_lookup[a['image_id']]
    assert a['segmentation']['size']==[i['height'],i['width']] and a['area']>0
amended.extend(additions)
assert [a for a in amended if a['image_id'] not in hp_lookup] == [
    a for a in test_original_data['annotations'] if a['image_id'] not in hp_lookup]
count_after = Counter(a['image_id'] for a in amended if a['image_id'] in hp_lookup)
assert set(count_after)==set(hp_lookup)
assert count_after == Counter({a['image_id']:a['after'] for a in HP_PATCH['summary']})
RAW['test'] = {**test_original_data, 'annotations':amended}
print('Revisi anotasi HP:', sum(a['before'] for a in HP_PATCH['summary']), '→', sum(count_after.values()), 'mask')
print('Status anotasi HP: revisi dibantu AI, menunggu konfirmasi pengguna di Roboflow.')


## 4. Cek gambar dan anotasi

Cek ukuran piksel dengan ukuran di JSON, sumber gambar dan hash piksel antarbagian, referensi ID, serta apakah semua mask bisa dibaca. Empat gambar dengan ukuran yang berbeda dipisahkan. Pembagian data awal dipakai untuk eksperimen.

Pengecekan duplikat memakai hash piksel. Audit hash punya keterbatasan untuk foto yang hanya mirip dan data pretraining dari luar proyek. Pada salinan COCO untuk evaluasi, luas dan kotak dihitung kembali dari mask. Polygon, RLE, lubang, dan bagian yang terpisah mengikuti anotasi awal.

In [ ]:
def encode_mask(mask):
    r = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    return {'size': [int(x) for x in r['size']], 'counts': r['counts'].decode('ascii')}
def decode_seg(seg, h, w):
    if isinstance(seg, list):
        polygons = [p for p in seg if len(p) >= 6 and len(p) % 2 == 0]
        if len(polygons) != len(seg) or not polygons: raise ValueError('Polygon kosong/tidak valid')
        r = mask_utils.merge(mask_utils.frPyObjects(polygons, h, w))
    elif isinstance(seg, dict):
        if list(seg['size']) != [h, w]: raise ValueError('Ukuran RLE tidak cocok')
        r = mask_utils.frPyObjects(seg, h, w) if isinstance(seg['counts'], list) else {
            'size': seg['size'], 'counts': seg['counts'].encode('ascii') if isinstance(seg['counts'], str) else seg['counts']}
    else: raise ValueError('Format segmentation tidak dikenali')
    mask = mask_utils.decode(r)
    if mask.ndim == 3: mask = np.any(mask, axis=2)
    return mask.astype(np.uint8)
def make_coco(data):
    c = COCO(); c.dataset = copy.deepcopy(data)
    with redirect_stdout(io.StringIO()): c.createIndex()
    return c

# Fingerprint semua anotasi dan seluruh file gambar: mencegah cache checkpoint dari dataset lain.
hasher = hashlib.sha256(b'cpmk3-canonical-v3-maskarea-quarantine')
file_records, dimension_bad, corrupt, hash_groups, source_groups = [], [], [], defaultdict(list), defaultdict(list)
for sp in ('train', 'valid', 'test'):
    data = RAW[sp]
    assert len({i['id'] for i in data['images']}) == len(data['images']), f'Duplicate image ID: {sp}'
    assert len({a['id'] for a in data['annotations']}) == len(data['annotations']), f'Duplicate annotation ID: {sp}'
    hasher.update(json.dumps(data, sort_keys=True).encode())
    for info in data['images']:
        f = (RAW_DIRS[sp] / info['file_name']).resolve()
        if not f.is_relative_to(RAW_DIRS[sp].resolve()): raise ValueError('Path gambar keluar folder')
        if not f.is_file(): raise FileNotFoundError(f)
        try:
            with Image.open(f) as pic:
                pic.load(); real = pic.size
                pixel_hash = hashlib.sha256(pic.convert('RGB').tobytes()).hexdigest()
        except Exception as exc:
            corrupt.append({'split':sp,'file':info['file_name'],'error':str(exc)}); continue
        digest = sha_file(f)
        record = {'split':sp,'image_id':info['id'],'file':info['file_name'],'source':source_name(info),
                  'sha256':digest,'pixel_sha256':pixel_hash,'width':real[0],'height':real[1],
                  'json_width':info['width'],'json_height':info['height'],'is_hp':is_hp(info)}
        hasher.update(digest.encode()); file_records.append(record)
        hash_groups[pixel_hash].append(record); source_groups[source_name(info)].append(record)
        if real != (info['width'], info['height']): dimension_bad.append(record)
if corrupt: raise RuntimeError(f'Gambar rusak: {corrupt[:5]}')
# Identitas sumber dan piksel sama lintas split -> hentikan, tidak membagi ulang secara acak.
leaks = [g for g in list(hash_groups.values()) + list(source_groups.values()) if len({r['split'] for r in g}) > 1]
if leaks: raise RuntimeError(f'Data leakage lintas split: {leaks[:3]}')
DATA_FINGERPRINT = hasher.hexdigest()
WORK = WORK_ROOT / DATA_FINGERPRINT[:16]
PROOF, METRICS, RUNS = WORK/'bukti', WORK/'metrik', WORK/'runs'
for folder in (PROOF,METRICS,RUNS): folder.mkdir(parents=True,exist_ok=True)
pd.DataFrame(file_records).to_csv(PROOF/'inventaris_gambar.csv',index=False)
pd.DataFrame(dimension_bad).to_csv(PROOF/'karantina_dimensi.csv',index=False)
write_json(PROOF/'duplicate_sources_within_split.json', [g for g in source_groups.values() if len(g)>1])
if dimension_bad and not QUARANTINE_DIMENSION_MISMATCH:
    raise ValueError('Perbaiki orientasi gambar/mask di Roboflow; jangan menebak resize otomatis.')
quarantined = {(r['split'],r['image_id']) for r in dimension_bad}
if any(sp == 'test' for sp,_ in quarantined):
    raise ValueError('Ada dimensi test salah. Perbaiki ekspor dahulu agar 40 HP/359 asli tetap utuh.')

# Konsistensi kelas diperiksa berdasarkan ID DAN nama pada seluruh split.
CAT_NAMES = {int(c['id']):c['name'] for c in RAW['train']['categories']}
assert len(set(CAT_NAMES.values())) == len(CAT_NAMES), 'Nama kategori duplikat'
for sp,d in RAW.items():
    assert {int(c['id']):c['name'] for c in d['categories']} == CAT_NAMES, f'Pemetaan kelas berbeda pada {sp}'
used_raw = {int(a['category_id']) for d in RAW.values() for a in d['annotations']}
PLACEHOLDER_IDS = {c['id'] for c in RAW['train']['categories']
                   if c['name']=='indoor-objects' and c.get('supercategory')=='none' and c['id'] not in used_raw}
CAT_IDS = sorted(set(CAT_NAMES)-PLACEHOLDER_IDS)
CAT2INDEX = {cid:k for k,cid in enumerate(CAT_IDS)}
INDEX2CAT = {k:cid for cid,k in CAT2INDEX.items()}
NC = len(CAT_IDS)
CATEGORIES = [copy.deepcopy(c) for c in RAW['train']['categories'] if c['id'] in CAT_IDS]
COCO_DATA, geometry_report, decode_errors = {}, [], []
for sp,d in RAW.items():
    images = [copy.deepcopy(i) for i in d['images'] if (sp,i['id']) not in quarantined]
    lookup = {i['id']:i for i in images}
    annotations = []
    for ann in d['annotations']:
        if ann['image_id'] not in {i['id'] for i in d['images']}: raise ValueError('Annotation image_id tidak ada')
        if ann['image_id'] not in lookup: continue
        info = lookup[ann['image_id']]
        try:
            if ann['category_id'] not in CAT2INDEX: raise ValueError('category_id tidak dikenal')
            m = decode_seg(ann.get('segmentation'), info['height'], info['width'])
            if not m.any(): raise ValueError('Mask kosong')
            rle = encode_mask(m)
            rb = {'size':rle['size'],'counts':rle['counts'].encode('ascii')}
            area = int(m.sum()); box = mask_utils.toBbox(rb).astype(float).tolist()
            a = copy.deepcopy(ann)
            # COCO export memberi area = luas bbox pada banyak mask. Hitung geometri yang benar dari mask.
            a.update(segmentation=rle, area=area, bbox=box, iscrowd=int(a.get('iscrowd',0)))
            annotations.append(a)
            geometry_report.append({'split':sp,'annotation_id':a['id'],
                                    'source_area':float(ann.get('area',0)),'mask_area':area,
                                    'area_changed':abs(float(ann.get('area',0))-area)>1})
        except Exception as exc: decode_errors.append({'split':sp,'annotation_id':ann['id'],'error':str(exc)})
    COCO_DATA[sp] = {'info':{'description':'COCO canonical: visible geometry preserved; mask area/bbox recomputed'},
                     'images':images,'annotations':annotations,'categories':CATEGORIES}
if decode_errors:
    write_json(PROOF/'mask_errors.json',decode_errors)
    raise ValueError(f'{len(decode_errors)} mask tidak dapat dibaca; perbaiki sebelum training.')
COCO_GT = {sp:make_coco(d) for sp,d in COCO_DATA.items()}
HP_IMAGES = [i for i in COCO_DATA['test']['images'] if is_hp(i)]
HP_IDS = {i['id'] for i in HP_IMAGES}
OFFICIAL_IDS = {i['id'] for i in COCO_DATA['test']['images'] if not is_hp(i)}
assert len(HP_IDS)==40 and len(OFFICIAL_IDS)==359, 'Identitas test tidak sesuai ekspor yang diaudit.'
assert {source_name(i) for i in HP_IMAGES} == HP_SOURCE_NAMES
assert not any(is_hp(i) for sp in ('train','valid') for i in COCO_DATA[sp]['images'])
train_freq = Counter(a['category_id'] for a in COCO_DATA['train']['annotations'])
valid_freq = Counter(a['category_id'] for a in COCO_DATA['valid']['annotations'])
hp_freq = Counter(a['category_id'] for a in COCO_DATA['test']['annotations'] if a['image_id'] in HP_IDS)
orig_freq = Counter(a['category_id'] for a in COCO_DATA['test']['annotations'] if a['image_id'] in OFFICIAL_IDS)
SEEN_CAT_IDS = sorted(train_freq)
COMMON_CAT_IDS = sorted(set(train_freq)&set(hp_freq)&set(orig_freq))
SCOPES = {'semua_kelas':CAT_IDS,'kelas_ada_di_train':SEEN_CAT_IDS,'kelas_sama_kedua_test':COMMON_CAT_IDS}
for sp,d in COCO_DATA.items(): write_json(PROOF/f'canonical_{sp}.json',d)
write_json(PROOF/'mapping_kelas.json', {'num_classes':NC,'placeholder_removed':sorted(PLACEHOLDER_IDS),
    'yolo_to_coco':INDEX2CAT,'maskrcnn_to_coco':{k+1:v for k,v in INDEX2CAT.items()}})
pd.DataFrame(geometry_report).to_csv(PROOF/'audit_geometri_mask.csv',index=False)
class_table = pd.DataFrame([{'category_id':c,'class':CAT_NAMES[c],'train':train_freq[c],
    'valid':valid_freq[c],'test_original':orig_freq[c],'test_hp':hp_freq[c]} for c in CAT_IDS])
class_table.to_csv(PROOF/'jumlah_instance_per_kelas.csv',index=False)
summary = {'fingerprint':DATA_FINGERPRINT,'raw_counts':CHOSEN['counts'],
    'effective_counts':{sp:len(d['images']) for sp,d in COCO_DATA.items()},
    'annotations':{sp:len(d['annotations']) for sp,d in COCO_DATA.items()},
    'original_test':len(OFFICIAL_IDS),'hp_test':len(HP_IDS),'classes':NC,
    'train_supported_classes':len(SEEN_CAT_IDS),'common_classes':len(COMMON_CAT_IDS),
    'hp_untrained_annotations':sum(v for c,v in hp_freq.items() if not train_freq[c]),
    'hp_untrained_classes':[CAT_NAMES[c] for c in hp_freq if not train_freq[c]],
    'quarantine_count':len(quarantined),'area_recomputed':sum(r['area_changed'] for r in geometry_report),
    'provenance_confirmed':SOURCE_V5_CONFIRMED,'provenance_note':SOURCE_V5_NOTE}
write_json(PROOF/'audit_summary.json',summary)
display(pd.DataFrame([{'split':sp,'images':len(d['images']),'instances':len(d['annotations'])} for sp,d in COCO_DATA.items()]))
print('Test asli:',len(OFFICIAL_IDS),'| HP:',len(HP_IDS),'| kelas:',NC,'| kelas ada di train:',len(SEEN_CAT_IDS))
print('Karantina dimensi:',len(quarantined),'| HP berkelas tanpa contoh train:',summary['hp_untrained_annotations'])
print('Status v5:', 'TERKONFIRMASI' if SOURCE_V5_CONFIRMED else 'BELUM TERKONFIRMASI - tidak boleh diklaim sebagai v5')


## 5. Lihat overlay dan cek 40 foto HP

Isi `review_manual_40_hp.csv` sambil melihat foto dan mask. Pilih `reviewed_all_objects=YA` setelah semua objek dalam kelas acuan sudah diperiksa. Untuk foto kelas, cek kursi, orang, meja, layar, dan area kelas acuan yang belum bermask. Untuk foto kasur, cek tempat tidur dan setiap bantal.

Cek definisi di sumber sebelum menyamakan nama kelas seperti `bagpack` dengan `backpack`, atau `screen` dengan `tv`. Kelas tanpa contoh latih diberi tanda. Untuk area seperti dinding atau lantai, ikuti cara anotasi proyek panoptic asal. Pakai overlay saat mengecek mask, lalu simpan tangkapan layar Roboflow sebagai bukti proses.

In [ ]:
def load_rgb(split, info):
    with Image.open(RAW_DIRS[split]/info['file_name']) as pic:
        arr = np.asarray(pic.convert('RGB')).copy()
    if arr.shape[:2] != (info['height'],info['width']): raise ValueError('Dimensi gambar berubah sejak audit.')
    return arr

def overlay_masks(split, info, predictions=None, score=FAIL_SCORE_THR):
    arr = load_rgb(split,info)
    anns = COCO_GT[split].imgToAnns[info['id']] if predictions is None else [p for p in predictions if p['score']>=score]
    for k,a in enumerate(anns):
        mask = decode_seg(a['segmentation'],info['height'],info['width']).astype(bool)
        if not mask.any(): continue
        rng = np.random.default_rng(k+17)
        color = rng.integers(55,240,3).astype(np.uint8)
        arr[mask] = (arr[mask]*0.6+color*0.4).astype(np.uint8)
        contours,_ = cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(arr,contours,-1,tuple(int(x) for x in color),3)
        yy,xx = np.where(mask)
        label = f"{k+1}:{CAT_NAMES[a['category_id']]}" + (f" {a['score']:.2f}" if predictions is not None else '')
        xy = (int(xx.min()),max(20,int(yy.min())))
        cv2.putText(arr,label,xy,cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,0,0),4,cv2.LINE_AA)
        cv2.putText(arr,label,xy,cv2.FONT_HERSHEY_SIMPLEX,0.65,(255,255,255),1,cv2.LINE_AA)
    return arr
GT_DIR = PROOF/'gt_overlay_HP'; GT_DIR.mkdir(exist_ok=True)
qa_rows = []
for info in sorted(HP_IMAGES,key=source_name):
    anns = COCO_GT['test'].imgToAnns[info['id']]
    image = Image.fromarray(overlay_masks('test',info))
    image.thumbnail((1200,1200)); image.save(GT_DIR/(Path(source_name(info)).stem+'.jpg'),quality=92)
    unsupported = sorted({CAT_NAMES[a['category_id']] for a in anns if not train_freq[a['category_id']]})
    qa_rows.append({'source_photo':source_name(info),'file_name':info['file_name'],'image_id':info['id'],
        'annotations':len(anns),'classes':'; '.join(CAT_NAMES[a['category_id']] for a in anns),
        'classes_without_train':'; '.join(unsupported),'reviewed_all_objects':'BELUM',
        'own_phone_confirmed':'BELUM','difficult_condition':'','annotation_screenshot':'','review_notes':''})
qa = pd.DataFrame(qa_rows)
qa.to_csv(PROOF/'audit_40_hp_otomatis.csv',index=False)
# Template tidak ditimpa supaya catatan manusia tidak hilang saat Run All ulang.
review_path = PROOF/'review_manual_40_hp.csv'
if not review_path.exists(): qa.to_csv(review_path,index=False)
# Lembar kontak hanya untuk preview, bukan bukti screenshot Roboflow.
for page,start in enumerate(range(0,len(qa_rows),10),1):
    group = qa_rows[start:start+10]
    fig,axes = plt.subplots(2,5,figsize=(18,11),squeeze=False)
    for ax,row in zip(axes.ravel(),group):
        with Image.open(GT_DIR/(Path(row['source_photo']).stem+'.jpg')) as im: ax.imshow(im)
        ax.set_title(f"{row['source_photo']}\n{row['annotations']} instance",fontsize=8);ax.axis('off')
    for ax in axes.ravel()[len(group):]:ax.axis('off')
    fig.suptitle('Ground truth untuk REVIEW - bukan prediksi model')
    fig.tight_layout();fig.savefig(PROOF/f'qa_hp_{page:02d}.jpg',dpi=120);plt.close(fig)
display(pd.DataFrame(HP_PATCH['summary'])[['photo','before','added','relabel','outside_reference','after']])
for evidence in ('perbaikan_per_foto.csv','riwayat_perbaikan_mask.csv'):
    shutil.copy2(PATCH_FILE.parent/evidence,PROOF/evidence)
shutil.copy2(PATCH_FILE,PROOF/PATCH_FILE.name)
write_json(PROOF/'coco_40_hp_revisi.json', {**RAW['test'],
 'images':[i for i in RAW['test']['images'] if is_hp(i)],
 'annotations':[a for a in RAW['test']['annotations'] if a['image_id'] in HP_IDS]})
print('Satu anotasi:',int((qa.annotations==1).sum()),'dari',len(qa),'| overlay:',GT_DIR)
display(qa[['source_photo','annotations','classes','classes_without_train']])
print('Periksa semua objek kelas acuan, bukan hanya objek yang sudah berwarna. Jangan menganggap area tanpa mask otomatis background yang benar.')


## Foto dengan kondisi sulit


In [ ]:
# Pengamatan dari foto asli, dengan status konfirmasi pemilik foto tetap pada checklist terpisah.
visible_conditions = {
    '20260713_111506.jpg': ('objek saling menutupi','Botol besar di depan menutupi sisi kursi.'),
    '20260714_105357.jpg': ('objek saling menutupi','Siswa di depan meja menutupi sebagian kursi dan siswa lain.'),
    '20260826_154026.jpg': ('objek saling menutupi','Beberapa orang dan sandaran kursi saling menutupi.'),
    '20260903_073220.jpg': ('objek saling menutupi','Kursi depan menutupi bagian kursi dan meja di belakang.'),
    '20260903_161001.jpg': ('sudut rendah','Kamera melihat ke depan dari bawah bagian atas sandaran kursi.'),
    '20260903_161212.jpg': ('sudut rendah dan terhalang','Kamera berada di belakang kursi, dengan orang di tepi kanan terpotong.'),
    '20261118_083050.jpg': ('backlight','Cahaya dari pintu kaca membuat kursi di depan tampak gelap.'),
    '20261118_084038.jpg': ('objek saling menutupi','Barisan kursi menutupi kursi lain, terutama dekat ujung lorong.'),
    '20261118_084046.jpg': ('backlight','Tirai dan jendela lebih terang daripada kursi di bawahnya.'),
    '20261118_084148.jpg': ('backlight','Jendela kaca blok terang dibanding lantai dan toilet.'),
    '20261118_192932.jpg': ('cahaya redup','Perbedaan intensitas kabel, TV, dan bagian dinding rendah.'),
    '20261118_192936.jpg': ('cahaya redup','Laptop dan sandaran tempat tidur tampak gelap.'),
    '20261118_192944.jpg': ('cahaya redup','Stopkontak dan barang di meja berada dalam pencahayaan rendah.'),
    '20261118_193001.jpg': ('cahaya redup','Ransel dan kabel di lantai sulit dibedakan dari bayangan.'),
    '20261118_193033.jpg': ('sudut menengadah','Kamera mengarah tajam ke plafon sehingga dinding dan kusen terlihat miring.'),
}
visible_rows = [
    {'source_photo':name, 'condition':condition,'description':detail}
    for name,(condition,detail) in visible_conditions.items() if name in HP_SOURCE_NAMES
]
condition_table = pd.DataFrame(visible_rows)
condition_table.to_csv(PROOF/'kondisi_sulit_terlihat.csv',index=False)
manual_photo_review = pd.read_csv(review_path,keep_default_na=False)
for name,(condition,detail) in visible_conditions.items():
    missing = (manual_photo_review.source_photo == name) & (manual_photo_review.difficult_condition == '')
    manual_photo_review.loc[missing,'difficult_condition'] = condition + ' — ' + detail
manual_photo_review.to_csv(review_path,index=False)
display(condition_table)


## Paket foto untuk dicek di Roboflow


In [ ]:
# Paket review privat untuk Roboflow. Nama JPEG dan dimensi persis seperti input mask.
hp_export = {
    'info':{'description':'Draf 40 foto HP dengan perbaikan mask dibantu AI untuk review di Roboflow'},
    'licenses':[],
    'images':[i for i in COCO_DATA['test']['images'] if i['id'] in HP_IDS],
    'annotations':[a for a in COCO_DATA['test']['annotations'] if a['image_id'] in HP_IDS],
}
used_hp_labels = {a['category_id'] for a in hp_export['annotations']}
hp_export['categories']=[c for c in CATEGORIES if c['id'] in used_hp_labels]
assert len(hp_export['images'])==40 and len(hp_export['annotations'])==352
assert len({i['file_name'] for i in hp_export['images']})==40
roboflow_bundle = WORK_ROOT/'Roboflow_Test40_Draf_Revisi.zip'
with zipfile.ZipFile(roboflow_bundle,'w') as z:
    z.writestr('_annotations.coco.json',json.dumps(hp_export,ensure_ascii=False),compress_type=zipfile.ZIP_DEFLATED)
    for image in hp_export['images']:
        name=Path(image['file_name']).name
        assert name==image['file_name']
        z.write(RAW_DIRS['test']/name,arcname=name)
with zipfile.ZipFile(roboflow_bundle) as z:
    assert len(z.namelist())==41 and z.testzip() is None
print('Paket review Roboflow tersedia di Output Kaggle:',roboflow_bundle.name)


## 6. Ubah data latih dan validasi ke format YOLO

Setiap objek menjadi satu baris dan memakai urutan ID kelas yang sama. Folder YOLO diisi data latih dan validasi. Mask pada ekspor yang dipakai berupa polygon satu bagian.

Kalau ekspor baru berisi RLE atau mask yang terpisah menjadi beberapa bagian, sel akan berhenti saat konversi berisiko menghilangkan bagian mask. Evaluasi kedua arsitektur memakai data COCO.

In [ ]:
# Train/Valid mempertahankan polygon asal jika ada. Evaluasi selalu memakai mask COCO canonical.
YOLO_DIR = WORK/'yolo_data'
YOLO_YAML = YOLO_DIR/'data.yaml'
conversion = []
for sp,ysp in [('train','train'),('valid','val')]:
    imdir,lbdir = YOLO_DIR/'images'/ysp,YOLO_DIR/'labels'/ysp
    imdir.mkdir(parents=True,exist_ok=True);lbdir.mkdir(parents=True,exist_ok=True)
    raw_ann = {a['id']:a for a in RAW[sp]['annotations']}
    for info in COCO_DATA[sp]['images']:
        dest = imdir/info['file_name']
        if not dest.exists(): shutil.copy2(RAW_DIRS[sp]/info['file_name'],dest)
        lines = []
        for ann in COCO_GT[sp].imgToAnns[info['id']]:
            src = raw_ann[ann['id']]; seg = src.get('segmentation')
            if ann.get('iscrowd',0):
                raise ValueError('YOLO polygon tidak mendukung crowd secara setara; putuskan kebijakan crowd dahulu.')
            if isinstance(seg,list) and len(seg)==1:
                pts = np.asarray(seg[0],dtype=np.float64).reshape(-1,2)
                method = 'original_polygon'
            else:
                # Jangan diam-diam mengambil kontur terbesar/mengisi lubang.
                mask = decode_seg(ann['segmentation'],info['height'],info['width'])
                contours,hierarchy = cv2.findContours(mask,cv2.RETR_TREE,cv2.CHAIN_APPROX_SIMPLE)
                if len(contours)!=1 or len(contours[0])<3:
                    raise ValueError(f"Konversi lossy untuk {sp}/{info['file_name']} ann {ann['id']}; perlu format training mask-preserving.")
                pts = contours[0].reshape(-1,2).astype(float);method='single_contour_from_mask'
            if len(pts)<3 or not np.isfinite(pts).all():raise ValueError('Titik polygon tidak valid')
            pts[:,0] = np.clip(pts[:,0]/info['width'],0,1)
            pts[:,1] = np.clip(pts[:,1]/info['height'],0,1)
            lines.append(' '.join([str(CAT2INDEX[ann['category_id']])]+[f'{v:.8f}' for v in pts.ravel()]))
            conversion.append({'split':sp,'annotation_id':ann['id'],'method':method,'components_dropped':0})
        (lbdir/(Path(info['file_name']).stem+'.txt')).write_text('\n'.join(lines)+'\n' if lines else '',encoding='utf-8')
spec = {'path':str(YOLO_DIR),'train':'images/train','val':'images/val',
        'names':{idx:CAT_NAMES[cid] for idx,cid in INDEX2CAT.items()}}
YOLO_YAML.write_text(yaml.safe_dump(spec,sort_keys=False,allow_unicode=True))
pd.DataFrame(conversion).to_csv(PROOF/'konversi_yolo.csv',index=False)
assert not (YOLO_DIR/'images'/'test').exists()
assert len(list((YOLO_DIR/'images'/'train').glob('*.jpg')))==len(COCO_DATA['train']['images'])
print('YOLO hanya berisi Train/Valid:',YOLO_YAML,'| test tidak disalin ke data training.')


## 7. Siapkan data dan model Mask R-CNN

Mask R-CNN membaca mask RLE dari salinan COCO. Kelas objeknya memakai ID 1 sampai K, dengan ID 0 untuk background. YOLO memakai ID 0 sampai K−1. FrozenBatchNorm dipakai saat pelatihan dan ketika bobot dimuat. Evaluasi memakai checkpoint yang tersedia.

Pada baseline, `min_size=max_size=640` membatasi sisi panjang gambar untuk Mask R-CNN. Model menambah padding sampai ukuran kelipatan 32. YOLO memakai letterbox persegi. Cara padding dan pengaturan pelatihan tiap arsitektur dicatat saat membandingkan hasil.

In [ ]:
class CocoMaskDataset(Dataset):
    def __init__(self,split):
        if split not in ('train','valid'): raise ValueError('Loader training hanya boleh train/valid.')
        self.split = split;self.images = COCO_DATA[split]['images']
    def __len__(self):return len(self.images)
    def __getitem__(self,idx):
        info = self.images[idx];arr = load_rgb(self.split,info);h,w = arr.shape[:2]
        boxes,masks,labels,areas,crowds = [],[],[],[],[]
        for a in COCO_GT[self.split].imgToAnns[info['id']]:
            mask = decode_seg(a['segmentation'],h,w)
            if not mask.any():continue
            yy,xx = np.where(mask)
            boxes.append([int(xx.min()),int(yy.min()),int(xx.max())+1,int(yy.max())+1])
            masks.append(mask);labels.append(CAT2INDEX[a['category_id']]+1)
            areas.append(float(mask.sum()));crowds.append(int(a.get('iscrowd',0)))
        target = {'boxes':torch.tensor(boxes,dtype=torch.float32).reshape(-1,4),
            'labels':torch.tensor(labels,dtype=torch.int64),
            'masks':torch.from_numpy(np.stack(masks)) if masks else torch.zeros((0,h,w),dtype=torch.uint8),
            'image_id':torch.tensor(info['id'],dtype=torch.int64),
            'area':torch.tensor(areas,dtype=torch.float32),'iscrowd':torch.tensor(crowds,dtype=torch.int64)}
        return to_tensor(arr),target

def detection_collate(batch):return tuple(zip(*batch))
def build_maskrcnn(pretrained=False,size=BASE_SIZE):
    # Arsitektur normalisasi identik saat train dan load. Tidak mengunduh bobot saat evaluasi.
    from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
    fpn = resnet_fpn_backbone(backbone_name='resnet50',weights=None,
                             norm_layer=FrozenBatchNorm2d,trainable_layers=3)
    model = torchvision.models.detection.MaskRCNN(fpn,num_classes=91,
        min_size=size,max_size=size,box_detections_per_img=PRED_MAX_DET,
        box_score_thresh=PRED_CONF,box_nms_thresh=0.7)
    if pretrained:
        weights = torchvision.models.detection.MaskRCNN_ResNet50_FPN_Weights.DEFAULT
        state = weights.get_state_dict(progress=True,check_hash=True)
        model.load_state_dict(state,strict=True)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,NC+1)
    model.roi_heads.mask_predictor = MaskRCNNPredictor(model.roi_heads.mask_predictor.conv5_mask.in_channels,256,NC+1)
    return model

def save_mask_checkpoint(path,model,epoch,steps):
    temp = Path(str(path)+'.tmp')
    torch.save({'state_dict':{k:v.detach().cpu() for k,v in model.state_dict().items()},
                'fingerprint':DATA_FINGERPRINT,'category_ids':CAT_IDS,'size':BASE_SIZE,
                'epoch':int(epoch),'steps':int(steps)},temp)
    temp.replace(path)
def load_mask_model(path,size=None):
    # weights_only=True; checkpoint hanya tensor dan tipe Python dasar.
    ck = torch.load(path,map_location='cpu',weights_only=True)
    assert ck['fingerprint']==TRAIN_BASE_FINGERPRINT and ck['category_ids']==CAT_IDS,'Checkpoint berasal dari data/kelas berbeda.'
    model = build_maskrcnn(pretrained=False,size=size or ck['size'])
    model.load_state_dict(ck['state_dict'],strict=True)
    return model.to(DEVICE).eval()
print('Contoh target Mask R-CNN:',{k:tuple(v.shape) for k,v in CocoMaskDataset('train')[0][1].items()})


## 8. Buat prediksi dan hitung skor mask

Sisi panjang foto dikecilkan sebelum masuk ke model supaya foto berukuran sampai 4080 piksel bisa diproses. Mask prediksi dikembalikan ke ukuran anotasi. Cara ini dipakai pada semua varian, dengan detail interpolasi yang tercatat.

Ambang skor untuk AP adalah `0.001`. Untuk analisis kegagalan, ambangnya `0.5`. Tiap model boleh memberi paling banyak 100 deteksi. Pengukuran waktu dimulai setelah model siap dan foto sudah dibaca. Waktu mencakup resize, inferensi, pemrosesan hasil, dan transfer data. Rata-rata dihitung dari panggilan setelah yang pertama.

In [ ]:
def resize_longside(arr,size):
    h,w = arr.shape[:2];scale = size/max(h,w)
    nw,nh = max(1,round(w*scale)),max(1,round(h*scale))
    return cv2.resize(arr,(nw,nh),interpolation=cv2.INTER_LINEAR)

def model_predict_image(model,kind,arr,size):
    # Inferensi pada gambar berukuran terbatas, lalu setiap mask diproyeksi balik satu per satu.
    # Ini menghindari tensor 100 x 4080 x 2296 di GPU ketika foto HP beresolusi tinggi.
    small = resize_longside(arr,size);h,w = arr.shape[:2];result = []
    if kind=='yolo':
        res = model.predict(cv2.cvtColor(small,cv2.COLOR_RGB2BGR),imgsz=size,
            device=0 if DEVICE.type=='cuda' else 'cpu',conf=PRED_CONF,iou=0.7,
            max_det=PRED_MAX_DET,retina_masks=True,rect=False,verbose=False)[0]
        if res.masks is not None:
            for box,mask in zip(res.boxes,res.masks.data):
                idx = int(box.cls.item());score = float(box.conf.item())
                if idx not in INDEX2CAT: raise ValueError('Output kelas YOLO tidak cocok dengan mapping.')
                m = mask.detach().cpu().numpy()
                full = cv2.resize(m.astype(np.uint8),(w,h),interpolation=cv2.INTER_NEAREST)
                if full.any():result.append({'category_id':INDEX2CAT[idx],'segmentation':encode_mask(full),'score':score})
        del res
    else:
        with torch.inference_mode(): output = model([to_tensor(small).to(DEVICE)])[0]
        labels = output['labels'].cpu().tolist();scores = output['scores'].cpu().tolist()
        for k,(lab,score) in enumerate(zip(labels,scores)):
            if score<PRED_CONF:continue
            if lab-1 not in INDEX2CAT:raise ValueError('Output kelas Mask R-CNN tidak cocok.')
            m = output['masks'][k,0].detach().float().cpu().numpy()
            full = (cv2.resize(m,(w,h),interpolation=cv2.INTER_LINEAR)>=0.5).astype(np.uint8)
            if full.any():result.append({'category_id':INDEX2CAT[lab-1],'segmentation':encode_mask(full),'score':float(score)})
        del output
    return result

def infer_split(model,kind,split,image_ids,size,save_dir=None):
    ids = set(image_ids);selected = [i for i in COCO_DATA[split]['images'] if i['id'] in ids]
    if len(selected)!=len(ids) or not selected:raise ValueError('Subset evaluasi tidak valid/kosong')
    rows,timings = [],[]
    for k,info in enumerate(selected):
        arr = load_rgb(split,info)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        start = time.perf_counter()
        pred = model_predict_image(model,kind,arr,size)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        elapsed = time.perf_counter()-start
        for p in pred:p['image_id']=int(info['id'])
        rows.extend(pred)
        timings.append({'image_id':int(info['id']),'seconds':elapsed,'n_pred':len(pred),'first_call':k==0})
        if (k+1)%50==0:print('Inferensi',split,k+1,'/',len(selected))
    if save_dir:
        save_dir.mkdir(parents=True,exist_ok=True)
        write_json(save_dir/'predictions_coco.json',rows)
        pd.DataFrame(timings).to_csv(save_dir/'inference_timing.csv',index=False)
    return rows,timings

def coco_evaluate(split,preds,ids,cats,verbose=False,gt_override=None):
    if not cats:raise ValueError('Scope kategori kosong')
    gt = COCO_GT[split] if gt_override is None else gt_override
    # bbox prediksi tidak dikirim: COCOeval segmentation menggunakan luas mask, bukan luas kotak.
    cleaned = [{'image_id':int(p['image_id']),'category_id':int(p['category_id']),
                'segmentation':copy.deepcopy(p['segmentation']),'score':float(p['score'])} for p in preds]
    with redirect_stdout(io.StringIO()):
        if cleaned:dt = gt.loadRes(cleaned)
        else:dt = make_coco({'info':{},'images':COCO_DATA[split]['images'],'categories':CATEGORIES,'annotations':[]})
        ev = COCOeval(gt,dt,'segm')
        ev.params.imgIds = sorted(ids);ev.params.catIds = sorted(cats)
        ev.params.maxDets = [1,10,100]
        ev.evaluate();ev.accumulate();ev.summarize()
    if verbose:print('AP50-95, AP50, AP75:',ev.stats[:3])
    return ev

def eval_metrics(ev):
    out = {}
    for key,index in [('mask_AP50_95',0),('mask_AP50',1),('mask_AP75',2),('mask_AR100',8)]:
        value = float(ev.stats[index]);out[key] = value if value>=0 else None
    return out

def eval_per_class(ev):
    rows=[]
    for k,c in enumerate(ev.params.catIds):
        p = ev.eval['precision'][:,:,k,0,-1];values = p[p>=0]
        rows.append({'category_id':c,'class':CAT_NAMES[c],'mask_AP50_95':float(values.mean()) if values.size else None,
                     'train_instances':train_freq[c],'gt_present':bool(values.size)})
    return rows

def load_run_model(kind,checkpoint,size):
    return YOLO(str(checkpoint)) if kind=='yolo' else load_mask_model(checkpoint,size)

def evaluate_checkpoint(kind,checkpoint,split,ids,run_name,size=BASE_SIZE,scopes=None):
    checkpoint = Path(checkpoint)
    if not checkpoint.is_file():raise FileNotFoundError(checkpoint)
    out = METRICS/run_name;out.mkdir(parents=True,exist_ok=True)
    key = {'dataset':DATA_FINGERPRINT,'checkpoint_sha256':sha_file(checkpoint),
           'image_ids':sorted(ids),'size':size,'conf':PRED_CONF,'max_det':PRED_MAX_DET,'preprocess':'longside-v1'}
    manifest = out/'prediction_manifest.json';pred_path = out/'predictions_coco.json'
    if manifest.is_file() and json.loads(manifest.read_text())==key and pred_path.is_file():
        preds = json.loads(pred_path.read_text());timings = pd.read_csv(out/'inference_timing.csv').to_dict('records')
    else:
        model = load_run_model(kind,checkpoint,size)
        try:preds,timings = infer_split(model,kind,split,ids,size,out)
        finally:
            del model;gc.collect()
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        write_json(manifest,key)
    active_scopes = scopes or SCOPES
    rows=[]
    for scope,cats in active_scopes.items():
        if not cats:continue
        ev = coco_evaluate(split,preds,ids,cats)
        per = eval_per_class(ev);pd.DataFrame(per).to_csv(out/f'AP_per_class_{scope}.csv',index=False)
        warm = [x['seconds'] for x in timings if not x.get('first_call',False)]
        rows.append({'run':run_name,'model':kind,'split':split,'scope':scope,'n_images':len(ids),
            'n_categories_requested':len(cats),'n_categories_with_gt':sum(r['gt_present'] for r in per),
            'size':size,'checkpoint':str(checkpoint),'seconds_per_image_excl_first':float(np.mean(warm)) if warm else None,
            'hp_labels_reviewed':HP_LABELS_REVIEWED,'source_v5_confirmed':SOURCE_V5_CONFIRMED,**eval_metrics(ev)})
    pd.DataFrame(rows).to_csv(out/'metrics.csv',index=False)
    return rows


## 9. Latih model dan simpan checkpoint

Set latih dan validasi ditetapkan sebelum percobaan. YOLO memilih checkpoint dari nilai fitness validasi bawaannya. Mask R-CNN memakai mask AP validasi. Skor kedua jenis model dihitung kembali dengan evaluasi COCO mask. Hasil uji dibaca setelah pilihan model dibuat dari validasi.

Saat dijalankan ulang, notebook bisa memakai run yang sudah selesai dengan konfigurasi serta data yang cocok. Log dan catatan progres membantu memeriksa waktu yang terpakai, termasuk saat sesi terputus.

In [ ]:
def utc():return datetime.now(timezone.utc).isoformat(timespec='seconds')
def append_csv(path,row):
    new = not path.exists()
    with path.open('a',newline='',encoding='utf-8') as f:
        writer = csv.DictWriter(f,fieldnames=list(row))
        if new:writer.writeheader()
        writer.writerow(row);f.flush()
TIME_LOG = PROOF/'durasi_training.csv'
TASK_LOG = PROOF/'logbook.csv'
def log(stage,status,note):append_csv(TASK_LOG,{'utc':utc(),'stage':stage,'status':status,'note':note})
def used_seconds(model_name):
    total = 0.; logged = set()
    if TIME_LOG.exists():
        table = pd.read_csv(TIME_LOG)
        subset = table[table.model==model_name]
        total = float(subset.seconds.sum()); logged = set(subset.run)
    # Bila kernel mati tanpa finally, heartbeat masih dihitung sebagai batas bawah waktu terpakai.
    for path in RUNS.glob('*/run_manifest.json'):
        run = path.parent.name
        meta = json.loads(path.read_text())
        hb = path.parent/'heartbeat.json'
        if meta.get('budget_key')==model_name and run not in logged and hb.is_file():
            total += float(json.loads(hb.read_text()).get('elapsed_seconds',0.))
    return total
def training_guard(model_name,minutes):
    if not SOURCE_V5_CONFIRMED or not SOURCE_V5_NOTE.strip():
        raise RuntimeError('Konfirmasi asal v5/bukti persetujuan dosen pada SOURCE_V5_CONFIRMED dan SOURCE_V5_NOTE dahulu. Audit tetap dapat dijalankan.')
    if DEVICE.type!='cuda':raise RuntimeError('Training utama memerlukan GPU Kaggle. Audit dapat berjalan di CPU.')
    remaining = MAX_SECONDS_PER_MODEL-used_seconds(model_name)-120
    if remaining<60:raise RuntimeError(f'Anggaran training {model_name} habis.')
    return min(float(minutes)*60,remaining)
def log_time(model_name,run_name,start,status):
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    elapsed = time.monotonic()-start
    append_csv(TIME_LOG,{'utc':utc(),'model':model_name,'run':run_name,'seconds':round(elapsed,3),
                         'status':status,'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'})
    log('training',status,f'{run_name}: {elapsed:.1f} detik wall-clock GPU, termasuk validasi/checkpoint dalam training call')

def run_meta(kind,size,minutes):
    return {'dataset':DATA_FINGERPRINT,'category_ids':CAT_IDS,'kind':kind,'size':size,
            'minutes':minutes,'seed':SEED,'epochs':EPOCHS,'ultralytics':importlib.metadata.version('ultralytics'),
            'batch':BATCH_MASK if kind=='maskrcnn' else BATCH_YOLO,'policy':'cpmk3-v3-no-random-split'}
def completed_checkpoint(run_dir,expected,checkpoint_name):
    manifest = run_dir/'run_manifest.json';complete = run_dir/'complete.json';ckpt = run_dir/checkpoint_name
    if manifest.exists() and json.loads(manifest.read_text())!=expected:
        raise RuntimeError(f'Konfigurasi run berubah di {run_dir}; gunakan output baru, jangan menimpa checkpoint lama.')
    if complete.is_file() and ckpt.is_file():return ckpt
    if manifest.exists():
        raise RuntimeError(f'Run {run_dir.name} belum selesai. Simpan bukti kegagalan; gunakan run ID baru untuk mengulang dan hitung waktu yang sudah dipakai.')
    return None

def train_yolo(run_name,weights,minutes,size,budget_key):
    folder = RUNS/run_name;meta = run_meta(weights,size,minutes);meta['budget_key']=budget_key
    existing = completed_checkpoint(folder,meta,'weights/best.pt')
    if existing:return existing
    seconds = training_guard(budget_key,minutes)
    seed_all();model=YOLO(weights)  # Unduh bobot sebelum penghitungan training.
    folder.mkdir(parents=True,exist_ok=True);write_json(folder/'run_manifest.json',meta)
    started = time.monotonic();status='failed'
    heartbeat = folder/'heartbeat.json'
    def beat(trainer):
        elapsed=time.monotonic()-started
        write_json(heartbeat,{'elapsed_seconds':elapsed,'epoch':int(trainer.epoch)+1,'utc':utc()})
        if elapsed>=seconds:trainer.stop=True
    model.add_callback('on_fit_epoch_end',beat)
    try:
        model.train(data=str(YOLO_YAML),imgsz=size,epochs=EPOCHS,batch=BATCH_YOLO,device=0,
            workers=WORKERS,seed=SEED,deterministic=True,project=str(RUNS),name=run_name,
            exist_ok=True,time=max(0.01,seconds/3600),patience=0,save=True,plots=True,
            amp=True,cache=False,multi_scale=False,rect=False,optimizer='SGD',lr0=0.01,
            momentum=0.937,weight_decay=0.0005,close_mosaic=0,
            hsv_h=0.015,hsv_s=0.7,hsv_v=0.4,degrees=0.0,translate=0.1,scale=0.5,
            shear=0.0,perspective=0.0,flipud=0.0,fliplr=0.5,mosaic=1.0,mixup=0.0,copy_paste=0.0)
        checkpoint=folder/'weights/best.pt'
        if not checkpoint.exists():raise RuntimeError('YOLO belum menghasilkan best.pt. Periksa results.csv dan error.')
        status='complete';write_json(folder/'complete.json',{'utc':utc(),'checkpoint':str(checkpoint)})
        return checkpoint
    finally:
        log_time(budget_key,run_name,started,status)
        del model;gc.collect();torch.cuda.empty_cache()

def train_mask(run_name='maskrcnn_640',minutes=MASK_MINUTES):
    folder=RUNS/run_name;meta=run_meta('maskrcnn',BASE_SIZE,minutes);meta['budget_key']='MaskRCNN'
    existing=completed_checkpoint(folder,meta,'best.pth')
    if existing:return existing
    seconds=training_guard('MaskRCNN',minutes)
    seed_all();model=build_maskrcnn(pretrained=True).to(DEVICE)
    loader=DataLoader(CocoMaskDataset('train'),batch_size=BATCH_MASK,shuffle=True,
        num_workers=WORKERS,collate_fn=detection_collate,generator=torch.Generator().manual_seed(SEED))
    optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=MASK_LR,momentum=0.9,weight_decay=0.0005)
    folder.mkdir(parents=True,exist_ok=True);write_json(folder/'run_manifest.json',meta)
    started=time.monotonic();status='failed';best_ap=-1.;step=0
    try:
        stop=False
        for epoch in range(EPOCHS):
            model.train();losses=[]
            for images,targets in loader:
                if time.monotonic()-started>=seconds:stop=True;break
                images=[x.to(DEVICE) for x in images]
                targets=[{k:v.to(DEVICE) for k,v in t.items()} for t in targets]
                # Warmup sederhana kemudian LR tetap; dicatat, tidak memilih LR memakai test.
                lr=MASK_LR*min(1.,(step+1)/100)
                for g in optimizer.param_groups:g['lr']=lr
                ld=model(images,targets);loss=sum(ld.values())
                if not torch.isfinite(loss):raise FloatingPointError('Loss Mask R-CNN nonfinite.')
                optimizer.zero_grad(set_to_none=True);loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),10.);optimizer.step();step+=1
                losses.append(float(loss.detach().cpu()))
                write_json(folder/'heartbeat.json',{'elapsed_seconds':time.monotonic()-started,'steps':step,'utc':utc()})
            if not losses:break
            save_mask_checkpoint(folder/'last.pth',model,epoch+1,step)
            # Validation GT asli/canonical, setiap epoch. Waktu validasi masuk anggaran.
            model.eval()
            predictions,_=infer_split(model,'maskrcnn','valid',{i['id'] for i in COCO_DATA['valid']['images']},BASE_SIZE)
            ev=coco_evaluate('valid',predictions,{i['id'] for i in COCO_DATA['valid']['images']},SEEN_CAT_IDS)
            ap=float(ev.stats[0])
            append_csv(folder/'maskrcnn_train.csv',{'epoch':epoch+1,'steps':step,'train_loss':float(np.mean(losses)),
                'valid_mask_AP50_95':ap,'elapsed_seconds':time.monotonic()-started,'lr':lr})
            if ap>best_ap:
                best_ap=ap;shutil.copy2(folder/'last.pth',folder/'best.pth')
            print(f'Mask R-CNN epoch {epoch+1}: loss={np.mean(losses):.4f}, validation AP={ap:.4f}')
            if stop or time.monotonic()-started>=seconds:break
        if not (folder/'best.pth').exists():raise RuntimeError('Belum ada checkpoint Mask R-CNN yang tervalidasi.')
        status='complete';write_json(folder/'complete.json',{'utc':utc(),'best_validation_AP':best_ap})
        return folder/'best.pth'
    finally:
        log_time('MaskRCNN',run_name,started,status)
        del model;gc.collect();torch.cuda.empty_cache()

write_json(PROOF/'environment.json',{'utc':utc(),'python':sys.version,'torch':torch.__version__,
    'torchvision':torchvision.__version__,'ultralytics':importlib.metadata.version('ultralytics'),
    'seed':SEED,'nrp_digit':NRP_LAST_DIGIT,'device':str(DEVICE),'source_v5_note':SOURCE_V5_NOTE,
    'fingerprint':DATA_FINGERPRINT,'training_enabled':RUN_TRAINING})
log('audit','completed','Data/geometry audited; semantik label HP masih membutuhkan review manusia.')


## Ambil checkpoint dari pelatihan di GPU


In [ ]:
# Gabungkan checkpoint dari pelatihan awal dan kedua notebook tambahan.
MODEL_RUN_PREFIX = {'YOLO11n':('yolo','yolo11n'), 'YOLO11s':('yolo','yolo11s'),
                    'MaskRCNN':('maskrcnn','maskrcnn')}
expected_runs = {f'{prefix}_{size}' for kind,prefix in MODEL_RUN_PREFIX.values() for size in RESOLUTIONS}
training_sources = []
for source_audit in INPUT_ROOT.rglob('audit_summary.json'):
    proof_dir = source_audit.parent
    if not (proof_dir/'durasi_training.csv').is_file() or not (proof_dir.parent/'runs').is_dir():
        continue
    if all((proof_dir/f'canonical_{sp}.json').is_file() and
           json.loads((proof_dir/f'canonical_{sp}.json').read_text())==COCO_DATA[sp] for sp in ('train','valid')):
        training_sources.append((source_audit, proof_dir))
assert len(training_sources)==3, f'Tiga sumber checkpoint yang cocok dibutuhkan. Ditemukan {len(training_sources)}.'
fingerprints = {json.loads(path.read_text())['fingerprint'] for path,_ in training_sources}
assert len(fingerprints)==1
TRAIN_BASE_FINGERPRINT = fingerprints.pop()
run_locations = {}
run_sources = {}
timings = []
original_reference = None
for source_audit,source_proof in training_sources:
    run_root = source_proof.parent
    assert run_root.name == TRAIN_BASE_FINGERPRINT[:16]
    original_labels = json.loads((source_proof/'canonical_test.json').read_text())
    assert original_labels['categories'] == COCO_DATA['test']['categories']
    assert original_labels['images'] == COCO_DATA['test']['images']
    assert [a for a in original_labels['annotations'] if a['image_id'] in OFFICIAL_IDS] == [
        a for a in COCO_DATA['test']['annotations'] if a['image_id'] in OFFICIAL_IDS]
    if original_reference is None:
        original_reference = original_labels
    else:
        assert original_reference == original_labels
    for run_dir in sorted((run_root/'runs').iterdir()):
        name = run_dir.name
        if name not in expected_runs or not (run_dir/'complete.json').is_file():
            continue
        assert name not in run_locations, f'Nama run bertabrakan: {name}'
        manifest = json.loads((run_dir/'run_manifest.json').read_text())
        assert manifest['dataset']==TRAIN_BASE_FINGERPRINT and manifest['category_ids']==CAT_IDS
        assert int(manifest['size'])==int(name.rsplit('_',1)[1])
        run_locations[name] = run_dir
        run_sources[name] = run_root
    timings.append(pd.read_csv(source_proof/'durasi_training.csv'))
assert set(run_locations)==expected_runs, f'Checkpoint ukuran belum lengkap: {sorted(expected_runs-set(run_locations))}'
time_table = pd.concat(timings,ignore_index=True).sort_values('utc')
assert set(time_table.loc[time_table.status=='complete','run'])==expected_runs
time_table.to_csv(TIME_LOG,index=False)
def weight_path(run_id):
    filename = 'best.pth' if run_id.startswith('maskrcnn_') else 'weights/best.pt'
    path = RUNS/run_id/filename
    assert path.is_file(), f'Bobot tidak ditemukan untuk {run_id}'
    return path

checkpoints = {}
for name,path in sorted(run_locations.items()):
    link = RUNS/name
    if link.is_symlink():
        assert link.resolve()==path.resolve()
    else:
        link.symlink_to(path,target_is_directory=True)
    ck = weight_path(name)
    checkpoints[name] = sha_file(ck)
write_json(PROOF/'checkpoint_source.json', {'dataset_before_HP_revision':TRAIN_BASE_FINGERPRINT,
    'dataset_after_HP_revision':DATA_FINGERPRINT, 'checkpoint_sha256':checkpoints})

# Prediksi validasi dan test asli pada bobot yang sama bisa dipakai kembali.
# Pengukuran COCO tetap dihitung ulang dengan definisi kelas yang dipakai di sini.
reuse = []
for model,(kind,prefix) in MODEL_RUN_PREFIX.items():
    for size in RESOLUTIONS:
        name = f'{prefix}_{size}'
        canonical_key = f'nrp_{name}_valid'
        aliases = [canonical_key]
        if model=='YOLO11n':
            aliases.append(f'nrp_{size}_valid')
        if size==BASE_SIZE:
            aliases.append(f'{model}_valid')
            reuse.extend([(name, f'{model}_valid', aliases, 'valid'),
                          (name, f'{model}_original_test', [f'{model}_original_test'], 'original')])
        reuse.append((name,canonical_key,aliases,'valid'))
for name,destination,aliases,partition in reuse:
    from_root = run_sources[name]/'metrik'
    source_dir = next((from_root/f for f in aliases if (from_root/f/'prediction_manifest.json').is_file()),None)
    if source_dir is None:
        continue
    manifest, preds, timing = (source_dir/f for f in ('prediction_manifest.json','predictions_coco.json','inference_timing.csv'))
    if not (preds.is_file() and timing.is_file()):
        continue
    meta = json.loads(manifest.read_text())
    wanted = OFFICIAL_IDS if partition=='original' else {i['id'] for i in COCO_DATA['valid']['images']}
    assert meta['dataset']==TRAIN_BASE_FINGERPRINT and meta['image_ids']==sorted(wanted)
    assert meta['checkpoint_sha256']==checkpoints[name]
    outdir = METRICS/destination
    outdir.mkdir(exist_ok=True)
    shutil.copy2(preds,outdir/preds.name)
    shutil.copy2(timing,outdir/timing.name)
    write_json(outdir/manifest.name,{**meta,'dataset':DATA_FINGERPRINT})
print('Muat',len(checkpoints),'checkpoint. Train, validation, kelas dan test asli cocok dengan semua run.')


## 10. Jalankan percobaan dan evaluasi

Mode audit dipakai untuk mengecek data. Pada mode pelatihan, model dilatih sesuai baseline dan ukuran yang diatur. Varian 640 memakai checkpoint baseline yang sama. Penilaian ukuran dilakukan pada set validasi. Perbandingan tiga model pada kedua set uji memakai baseline 640.

Hasil uji asli dan uji HP dicatat terpisah. Setiap baris menyertakan status pengecekan label dan asal dataset.

In [ ]:
YOLO_N_CKPT=RUNS/'yolo11n_640/weights/best.pt'
YOLO_S_CKPT=RUNS/'yolo11s_640/weights/best.pt'
MASK_CKPT=RUNS/'maskrcnn_640/best.pth'
if RUN_TRAINING:
    train_yolo('yolo11n_640','yolo11n-seg.pt',YOLO_N_MINUTES,640,'YOLO11n')
    train_yolo('yolo11s_640','yolo11s-seg.pt',YOLO_S_MINUTES,640,'YOLO11s')
    train_mask()
    if RUN_NRP_EXPERIMENT:
        for sz in (320,960):train_yolo(f'yolo11n_{sz}','yolo11n-seg.pt',NRP_MINUTES,sz,'YOLO11n')
else:
    print('Training dijalankan di notebook CPMK3 Training GPU. Sesi ini memproses revisi anotasi dan hasil yang tersedia.')
BASELINES=[('YOLO11n','yolo',YOLO_N_CKPT),('YOLO11s','yolo',YOLO_S_CKPT),('MaskRCNN','maskrcnn',MASK_CKPT)]
VAL_IDS={i['id'] for i in COCO_DATA['valid']['images']}
matrix_rows, history_by_run = [], {}
for model,(kind,prefix) in MODEL_RUN_PREFIX.items():
    for size in RESOLUTIONS:
        run_name = f'{prefix}_{size}'
        ck = weight_path(run_name)
        history_path = RUNS/run_name/('maskrcnn_train.csv' if kind=='maskrcnn' else 'results.csv')
        history = pd.read_csv(history_path)
        history.columns = history.columns.str.strip()
        metric = ('valid_mask_AP50_95' if kind=='maskrcnn' else
                  next(column for column in history.columns if 'mAP50-95' in column and '(M)' in column))
        history_by_run[(model,size)] = history.set_index('epoch')[metric]
        rows = evaluate_checkpoint(kind,ck,'valid',VAL_IDS,f'nrp_{run_name}_valid',size=size,
                                   scopes={'kelas_ada_di_train':SEEN_CAT_IDS})
        duration = float(time_table.loc[time_table.run==run_name,'seconds'].sum())
        assert len(rows)==1
        matrix_rows.append({**rows[0],'model_name':model,'nrp_digit':NRP_LAST_DIGIT,
                           'input_size':size,'epochs_reached':len(history),'training_seconds':duration})
nrp_df = pd.DataFrame(matrix_rows)
assert len(nrp_df)==9 and all(set(group.input_size)==set(RESOLUTIONS) for _,group in nrp_df.groupby('model_name'))
nrp_df.to_csv(METRICS/'nrp_0_validation.csv',index=False)
display(nrp_df[['model_name','input_size','mask_AP50_95','mask_AP50','epochs_reached','training_seconds']])
best = nrp_df.sort_values(['model_name','mask_AP50_95','input_size'],ascending=[True,False,True]).groupby('model_name').first()
write_json(METRICS/'input_terpilih_dari_validation.json',[
    {'model':name,'size':int(row.input_size),'validation_AP':float(row.mask_AP50_95)} for name,row in best.iterrows()
])

# Grafik kedua membandingkan titik epoch yang sama dalam setiap keluarga/model.
epoch_rows = []
for model,(kind,prefix) in MODEL_RUN_PREFIX.items():
    histories = {size:history_by_run[(model,size)] for size in RESOLUTIONS}
    shared_epochs = set.intersection(*(set(h.index) for h in histories.values()))
    assert shared_epochs
    last_common = max(shared_epochs)
    for size,h in histories.items():
        epoch_rows.append({'model_name':model,'input_size':size,'common_epoch':int(last_common),
                           'mask_AP50_95_logged':float(h.loc[last_common]),
                           'validation_metric_backend':'COCOeval' if kind=='maskrcnn' else 'Ultralytics'})
epoch_nrp = pd.DataFrame(epoch_rows)
epoch_nrp.to_csv(METRICS/'nrp_0_validation_epoch_setara.csv',index=False)
display(epoch_nrp)

comparison=[]
if RUN_EVALUATION:
    for name,kind,ck in BASELINES:
        if not ck.is_file():
            print('Checkpoint belum tersedia:',name);continue
        # Validasi model baseline juga disimpan untuk dokumentasi; test tidak menentukan checkpoint.
        evaluate_checkpoint(kind,ck,'valid',VAL_IDS,f'{name}_valid',scopes={'kelas_ada_di_train':SEEN_CAT_IDS})
        for subset,ids in [('original',OFFICIAL_IDS),('hp',HP_IDS)]:
            for row in evaluate_checkpoint(kind,ck,'test',ids,f'{name}_{subset}_test'):
                row.update(model_name=name,test_set=subset)
                comparison.append(row)
comp=pd.DataFrame(comparison)
if not comp.empty:
    comp.to_csv(METRICS/'perbandingan_3_model_2_test.csv',index=False)
    visible_results = comp.loc[comp.scope=='semua_kelas', ['model_name','test_set','n_images','mask_AP50','mask_AP50_95']].copy()
    for metric in ('mask_AP50','mask_AP50_95'):
        visible_results[metric] = (100*visible_results[metric]).round(2)
    visible_results = visible_results.rename(columns={'model_name':'Model','test_set':'Data uji','n_images':'Foto','mask_AP50':'Mask AP50 (%)','mask_AP50_95':'Mask AP50–95 (%)'})
    print('Hasil utama tiga model pada test asli dan foto HP hasil revisi')
    display(visible_results)
    print('Tabel lengkap memuat perbandingan tambahan dengan kelas yang sama di kedua test set.')
    display(comp[['model_name','test_set','scope','n_images','mask_AP50_95','mask_AP50']])
    if not HP_LABELS_REVIEWED:print('Label HP hasil revisi bantuan AI masih menunggu konfirmasi di Roboflow. Interpretasi AP HP bersifat sementara.')


## Bandingkan skor sebelum dan sesudah revisi label HP


In [ ]:
# Bandingkan label awal dan revisi pada model serta kelas yang sama.
reference_hp_gt = make_coco(original_reference)
reference_classes = {a['category_id'] for a in original_reference['annotations'] if a['image_id'] in HP_IDS}
review_classes = set(hp_freq)
shared_hp_classes = sorted(reference_classes & review_classes & set(SEEN_CAT_IDS))
assert shared_hp_classes
revision_impact = []
for name,kind,ck in BASELINES:
    pred_path = METRICS/f'{name}_hp_test/predictions_coco.json'
    if not pred_path.is_file():
        continue
    pred = json.loads(pred_path.read_text())
    before = eval_metrics(coco_evaluate('test',pred,HP_IDS,shared_hp_classes,gt_override=reference_hp_gt))
    after = eval_metrics(coco_evaluate('test',pred,HP_IDS,shared_hp_classes))
    revision_impact.append({'model':name,'matched_scope_classes':len(shared_hp_classes),
        'mask_AP50_95_before':before['mask_AP50_95'],'mask_AP50_95_revised':after['mask_AP50_95'],
        'change_percentage_points':100*(after['mask_AP50_95']-before['mask_AP50_95']),
        'mask_AP50_before':before['mask_AP50'],'mask_AP50_revised':after['mask_AP50']})
if revision_impact:
    impact = pd.DataFrame(revision_impact)
    impact.to_csv(METRICS/'dampak_revisi_anotasi_hp.csv',index=False)
    print('Skor mask dengan prediksi yang sama pada kelas yang sama di kedua revisi label HP.')
    display(impact)


# Hipotesis sebelum run tambahan NRP 0

Catatan awal dibuat pada 23 November 2026 pukul 20:38:48 WIB, sebelum percobaan tambahan YOLO11s dan Mask R-CNN dikirim ke Kaggle. Rubrik baru diterima saat pelatihan awal sudah berjalan. Waktu itu, YOLO11n dan YOLO11s pada 640 piksel sudah punya hasil validasi awal. Mask R-CNN 640 masih dilatih. Catatan 17–19 November mencakup pengambilan foto, anotasi, pembagian data, dan persiapan kode.

- Pada 960 piksel, dugaan saya YOLO11s akan unggul pada kelas **book** dan **plug**. Input yang lebih besar membantu menjaga detail benda kecil, dan YOLO11s punya kapasitas model lebih besar dibanding YOLO11n.
- Saya menduga Mask R-CNN cukup bagus untuk kelas **chair** dan **pillow**, terutama kalau objeknya berdekatan. Model ini membuat mask per area objek, sehingga bisa membantu membedakan tiap objek yang berhimpitan. Grid mask 28×28 masih bisa kehilangan bagian tipis seperti kaki kursi.
- Dari ukuran awal 320 piksel, saya menduga kenaikan resolusi paling terasa pada objek kecil. Permukaan besar seperti **wall-other-merged** dipakai sebagai pembanding. Dugaan ini dicek lewat AP tiap kelas di set validasi dan jumlah anotasi yang tersedia.

Tiga ukuran pada tiap model memakai pembagian data, seed, dan pengaturan pelatihan yang sama. Tabel utama memakai checkpoint terbaik di validasi dengan anggaran waktu tiap model. Log yang tersedia juga dipakai untuk melihat skor pada jumlah epoch yang sama. Foto HP dipakai untuk uji akhir dan melihat selisih skor saat model bertemu foto ruangan lain.

Urutan pengerjaan bisa dicek dari waktu commit awal catatan ini dan waktu mulai pelatihan tambahan di log Kaggle.


In [ ]:
# Gunakan AP mask per kelas dari validation untuk memeriksa hipotesis tambahan.
focus_classes = ['book','plug','chair','pillow','wall-other-merged']
hypothesis_rows = []
for name,(kind,prefix) in MODEL_RUN_PREFIX.items():
    for size in RESOLUTIONS:
        report_file = METRICS/f'nrp_{prefix}_{size}_valid/AP_per_class_kelas_ada_di_train.csv'
        class_ap = pd.read_csv(report_file).set_index('class')
        for category in focus_classes:
            value = class_ap.at[category,'mask_AP50_95'] if category in class_ap.index else None
            hypothesis_rows.append({'model':name,'input_size':size,'class':category,'mask_AP50_95':value})
hypothesis_table = pd.DataFrame(hypothesis_rows)
hypothesis_table.to_csv(METRICS/'uji_hipotesis_AP_per_kelas.csv',index=False)
forms = hypothesis_table.pivot(index=['class','input_size'],columns='model',values='mask_AP50_95')
observations = []
for category in ('book','plug'):
    measured = forms.loc[(category,960)].dropna()
    winner = ', '.join(sorted(measured[measured==measured.max()].index)) if len(measured) else 'GT belum tersedia'
    observations.append({'hypothesis':f'YOLO11s unggul pada {category} di 960',
                         'evidence':f'AP tertinggi di validation: {winner}'})
for category in ('chair','pillow'):
    measured = forms.loc[(category,960)].dropna()
    found = 'GT belum tersedia'
    if 'MaskRCNN' in measured:
        rank = int((measured>measured['MaskRCNN']).sum()+1)
        found = f'Peringkat AP Mask R-CNN di 960: {rank} dari {len(measured)}'
    observations.append({'hypothesis':f'Mask R-CNN kompetitif pada {category}','evidence':found})
for name in MODEL_RUN_PREFIX:
    small_gain = [forms.loc[(c,960),name]-forms.loc[(c,320),name] for c in ('book','plug')
                  if pd.notna(forms.loc[(c,960),name]) and pd.notna(forms.loc[(c,320),name])]
    wall_gain = forms.loc[('wall-other-merged',960),name]-forms.loc[('wall-other-merged',320),name]
    if small_gain and pd.notna(wall_gain):
        note = f'Rata-rata selisih AP {len(small_gain)} kelas kecil {100*np.mean(small_gain):+.2f} pp. Dinding {100*wall_gain:+.2f} pp.'
    else:
        note = 'Ground truth di validation tidak mencakup semua kelas untuk perbandingan ini.'
    observations.append({'hypothesis':f'Ukuran besar lebih membantu objek kecil pada {name}','evidence':note})
hypothesis_observations = pd.DataFrame(observations)
hypothesis_observations.to_csv(METRICS/'pembahasan_hipotesis_tambahan.csv',index=False)
display(hypothesis_observations)
display(forms)


## 11. Lihat grafik dan beda skor uji

Selisih dihitung dari AP uji HP dikurangi AP uji asli. Kalikan 100 untuk membacanya dalam poin persentase. Nilai negatif berarti skor pada foto HP lebih rendah. Saat membaca selisihnya, perhatikan juga komposisi kelas dan kelengkapan anotasi di kedua set uji. Grafik loss diambil dari log pelatihan yang tersedia.

In [ ]:
if not comp.empty:
    shifts=[]
    for scope in comp.scope.unique():
        p=comp[comp.scope==scope].pivot(index='model_name',columns='test_set',values='mask_AP50_95')
        if {'original','hp'}.issubset(p.columns):
            p['delta_hp_minus_original']=p['hp']-p['original']
            p['delta_percentage_points']=100*p['delta_hp_minus_original']
            p=p.reset_index();p['scope']=scope;shifts.append(p)
    if shifts:
        shift=pd.concat(shifts,ignore_index=True);shift.to_csv(METRICS/'domain_shift.csv',index=False);display(shift)
    fig,axes=plt.subplots(1,2,figsize=(13,5))
    for ax,scope in zip(axes,['semua_kelas','kelas_sama_kedua_test']):
        table=comp[comp.scope==scope].pivot(index='model_name',columns='test_set',values='mask_AP50_95')
        if not table.empty:table.plot.bar(ax=ax,ylim=(0,1),rot=0,title=scope)
        ax.set_ylabel('Mask AP@[.50:.95] (0-1)')
    fig.suptitle('Test asli vs HP - provisional jika label/provenance belum disahkan')
    fig.tight_layout();fig.savefig(METRICS/'perbandingan_mask_AP.png',dpi=150);plt.show()
if not nrp_df.empty:
    fig,ax = plt.subplots(figsize=(8,4.8))
    for name,group in nrp_df.sort_values('input_size').groupby('model_name'):
        ax.plot(group.input_size,100*group.mask_AP50_95,'o-',label=name)
    ax.set(xticks=list(RESOLUTIONS),xlabel='Ukuran input',ylabel='Validation mask AP50–95 (%)',
           title='Eksperimen NRP 0 pada tiga model')
    ax.legend()
    ax.grid(alpha=.2)
    fig.tight_layout()
    fig.savefig(METRICS/'nrp_input_size.png',dpi=150)
    plt.show()
    fig,axes = plt.subplots(1,3,figsize=(12,4),squeeze=False)
    for ax,(name,group) in zip(axes.ravel(),epoch_nrp.groupby('model_name')):
        table = group.sort_values('input_size')
        ax.plot(table.input_size,100*table.mask_AP50_95_logged,'o-')
        ax.set(xticks=list(RESOLUTIONS),xlabel='Ukuran input',title=f'{name} · epoch {table.common_epoch.iloc[0]}')
        ax.grid(alpha=.2)
    axes.ravel()[0].set_ylabel('Validation mask AP50–95 dari log (%)')
    fig.suptitle('Perbandingan ukuran pada epoch yang sama dalam tiap model')
    fig.tight_layout()
    fig.savefig(METRICS/'nrp_input_size_epoch_setara.png',dpi=150)
    plt.show()

# Kurva loss asli tersedia setelah training, tidak dibuat dari angka contoh.
fig,axes=plt.subplots(1,3,figsize=(15,4));plotted=False
for ax,(name,kind,ck) in zip(axes,BASELINES):
    f=ck.parent.parent/'results.csv' if kind=='yolo' else ck.parent/'maskrcnn_train.csv'
    if f.is_file():
        h=pd.read_csv(f);h.columns=h.columns.str.strip()
        cols=[c for c in h.columns if c.startswith('train/') and 'loss' in c] if kind=='yolo' else ['train_loss']
        for c in cols:ax.plot(np.arange(1,len(h)+1),h[c],label=c)
        if cols:ax.legend(fontsize=7);plotted=True
    ax.set(title=name,xlabel='Epoch tercatat',ylabel='Loss training')
if plotted:fig.tight_layout();fig.savefig(METRICS/'kurva_loss_training.png',dpi=150);plt.show()
else:plt.close(fig)
print('Baca domain shift pada scope kelas sama; perbedaan label-space dan kelengkapan GT tetap merupakan confounder.')


## 12. Cek lima contoh kegagalan

Foto diurutkan dari jumlah objek yang terlewat dan prediksi berlebih, yaitu `FN + FP`. Pencocokan mempertimbangkan kelas yang sama, IoU mask, dan skor minimal 0,5. Notebook mengambil lima foto dengan kesalahan terbanyak. Catat apa yang terlihat di CSV agar catatan bisa dibaca lagi saat notebook dijalankan ulang.

Bobot yang sama dipakai untuk mencoba ukuran 320, 640, dan 960 pada foto yang sama. Hasilnya menunjukkan pengaruh ukuran inferensi secara keseluruhan. Untuk membahas faktor tertentu seperti cahaya, objek tertutup, atau grid prototype YOLO, gunakan percobaan yang mengontrol faktor itu. Pada kelas tanpa contoh latih, catat keterbatasan datanya.

In [ ]:
def mask_matches(info,anns,preds,score=FAIL_SCORE_THR,iou_threshold=0.5):
    # Diagnostik class-aware dan score-ordered (bukan pengganti COCO AP).
    preds=sorted([p for p in preds if p['score']>=score],key=lambda x:-x['score'])
    gt_rle=[a['segmentation'] for a in anns]
    pred_rle=[p['segmentation'] for p in preds]
    def rb(seg):return {'size':seg['size'],'counts':seg['counts'].encode('ascii') if isinstance(seg['counts'],str) else seg['counts']}
    ious=mask_utils.iou([rb(r) for r in pred_rle],[rb(r) for r in gt_rle],[0]*len(anns)) if preds and anns else np.zeros((len(preds),len(anns)))
    matched=set();tp=0;matched_ious=[]
    for p_idx,p in enumerate(preds):
        choices=[g for g,a in enumerate(anns) if g not in matched and a['category_id']==p['category_id']]
        if not choices:continue
        g=max(choices,key=lambda g:ious[p_idx,g])
        if ious[p_idx,g]>=iou_threshold:matched.add(g);tp+=1;matched_ious.append(float(ious[p_idx,g]))
    unseen=sum(not train_freq[a['category_id']] for a in anns)
    return {'TP':tp,'FN':len(anns)-tp,'FP':len(preds)-tp,'n_gt':len(anns),'n_pred':len(preds),
            'untrained_gt':unseen,'mean_matched_IoU':float(np.mean(matched_ious)) if matched_ious else None}
FAIL_CSV=METRICS/'5_kasus_kegagalan_HP.csv'
FAIL_IDS=set()
pred_file=METRICS/'YOLO11n_hp_test/predictions_coco.json'
if RUN_FAILURE_ANALYSIS and pred_file.is_file():
    by=defaultdict(list)
    for p in json.loads(pred_file.read_text()):by[p['image_id']].append(p)
    ranked=[]
    for info in HP_IMAGES:
        anns=COCO_GT['test'].imgToAnns[info['id']]
        row=mask_matches(info,anns,by[info['id']]);row.update(image_id=info['id'],source_photo=source_name(info))
        row['total_error']=row['FN']+row['FP'];ranked.append(row)
    ranked=pd.DataFrame(ranked).sort_values(['total_error','FN','image_id'],ascending=[False,False,True])
    ranked.to_csv(METRICS/'ranking_kegagalan_HP.csv',index=False)
    selected=ranked[ranked.total_error>0].head(5).copy()
    FAIL_IDS=set(int(x) for x in selected.image_id)
    # Jangan menimpa tulisan analisis manusia pada rerun.
    if FAIL_CSV.is_file():
        prior=pd.read_csv(FAIL_CSV).set_index('image_id')
    else:prior=pd.DataFrame()
    for col in ['apa_yang_gagal','hipotesis_mekanisme','bukti_eksperimen','kesimpulan_dan_batasan']:
        selected[col]=[prior.at[i,col] if not prior.empty and i in prior.index and col in prior else '' for i in selected.image_id]
    selected.to_csv(FAIL_CSV,index=False);display(selected)
    for info in HP_IMAGES:
        if info['id'] not in FAIL_IDS:continue
        fig,axes=plt.subplots(1,3,figsize=(15,7))
        for ax,title,arr in zip(axes,['Foto asli','Ground truth','Prediksi YOLO11n (score >= 0.5)'],
              [load_rgb('test',info),overlay_masks('test',info),overlay_masks('test',info,by[info['id']])]):
            ax.imshow(arr);ax.axis('off');ax.set_title(title)
        fig.suptitle(source_name(info));fig.tight_layout()
        fig.savefig(METRICS/f"failure_{info['id']}.jpg",dpi=130);plt.close(fig)
    if len(FAIL_IDS)<5:print('Kurang dari lima foto dengan FN/FP > 0. Periksa kualitas mask lain; jangan mengarang kegagalan.')
    print('Objek kelas tanpa contoh train adalah keterbatasan label-space, bukan bukti kegagalan resolusi prototype.')
else:print('Analisis kegagalan menunggu prediksi baseline YOLO11n pada 40 HP.')

# Uji penjelasan resolusi: bobot SAMA, lima gambar SAMA, hanya ukuran inferensi berubah.
small_rows=[]
if RUN_FAILURE_ANALYSIS and FAIL_IDS and YOLO_N_CKPT.is_file():
    for size in RESOLUTIONS:
        out=METRICS/f'failure_probe_{size}'
        model=load_run_model('yolo',YOLO_N_CKPT,size)
        try:preds,_=infer_split(model,'yolo','test',FAIL_IDS,size,out)
        finally:
            del model;gc.collect()
            if DEVICE.type=='cuda':torch.cuda.empty_cache()
        by=defaultdict(list)
        for p in preds:by[p['image_id']].append(p)
        for info in HP_IMAGES:
            if info['id'] not in FAIL_IDS:continue
            row=mask_matches(info,COCO_GT['test'].imgToAnns[info['id']],by[info['id']])
            row.update(image_id=info['id'],source_photo=source_name(info),input_size=size,
                       fixed_checkpoint_sha256=sha_file(YOLO_N_CKPT));small_rows.append(row)
        ev=coco_evaluate('test',preds,FAIL_IDS,CAT_IDS)
        write_json(out/'subset_ap.json',eval_metrics(ev))
    probe=pd.DataFrame(small_rows);probe.to_csv(METRICS/'eksperimen_kecil_per_foto.csv',index=False)
    display(probe[['source_photo','input_size','TP','FN','FP','mean_matched_IoU','untrained_gt']])
    print('Perubahan ukuran inferensi menguji sensitivitas resolusi secara keseluruhan. Ini tidak mengisolasi prototype mask dari backbone/NMS dan tidak membuktikan kausalitas tunggal.')


## Catatan dari lima kasus kegagalan


In [ ]:
# Catat yang terukur pada lima foto dan uji pengaruh ambang skor dengan prediksi yang sama.
if FAIL_IDS:
    diagnostics = []
    reports = pd.read_csv(FAIL_CSV).set_index('image_id')
    for field in ('apa_yang_gagal','hipotesis_mekanisme','bukti_eksperimen','kesimpulan_dan_batasan'):
        reports[field] = reports[field].astype('string').fillna('')
    yh = defaultdict(list)
    pred_path = METRICS/'YOLO11n_hp_test/predictions_coco.json'
    for row in json.loads(pred_path.read_text()):
        if row['image_id'] in FAIL_IDS:
            yh[row['image_id']].append(row)
    resolution_evidence = pd.read_csv(METRICS/'eksperimen_kecil_per_foto.csv')
    for info in HP_IMAGES:
        image_id = int(info['id'])
        if image_id not in FAIL_IDS:
            continue
        ground_truth = COCO_GT['test'].imgToAnns[image_id]
        predicted = yh[image_id]
        def rb(seg):
            return {'size':seg['size'],'counts':seg['counts'].encode('ascii') if isinstance(seg['counts'],str) else seg['counts']}
        ious = mask_utils.iou([rb(p['segmentation']) for p in predicted],
                             [rb(a['segmentation']) for a in ground_truth],[0]*len(ground_truth)) if predicted else np.zeros((0,len(ground_truth)))
        print_classes = Counter()
        for j,gt in enumerate(ground_truth):
            category = gt['category_id']
            same_class = [i for i,p in enumerate(predicted) if p['category_id']==category]
            high_iou = [i for i in same_class if ious[i,j]>=0.5]
            strongest = max((predicted[i]['score'] for i in high_iou), default=0.)
            if strongest>=FAIL_SCORE_THR:
                continue
            class_name = CAT_NAMES[category]
            if strongest>0:
                issue = 'skor_deteksi_rendah'
            elif any(ious[i,j]>=0.5 and p['score']>=FAIL_SCORE_THR
                     for i,p in enumerate(predicted) if p['category_id']!=category):
                issue = 'kelas_prediksi_berbeda'
            elif same_class:
                issue = 'mask_kelas_sama_IoU_di_bawah_0.5'
            else:
                issue = 'tidak_ada_kandidat_kelas_sama'
            print_classes[(class_name, issue)] += 1
            diagnostics.append({'source_photo':source_name(info),'image_id':image_id,'gt_annotation_id':gt['id'],
                'class':class_name,'event':issue,
                'best_IoU_same_class':max((float(ious[i,j]) for i in same_class),default=0.),
                'max_confidence_same_class_IoU50':strongest})
        text = '; '.join(f'{count} {label} ({issue.replace("_"," ")})'
                         for (label,issue),count in print_classes.most_common(4))
        row = reports.loc[image_id]
        reports.at[image_id,'apa_yang_gagal'] = (
            f"Pada confidence 0,50, FN={int(row.FN)} dan FP={int(row.FP)}. {text}"
        )
        low = mask_matches(info,ground_truth,predicted,score=0.25)
        at_sizes = resolution_evidence[resolution_evidence.image_id==image_id].sort_values('input_size')
        resolution_text = '; '.join(f"{r.input_size}px menghasilkan {r.TP} TP, {r.FN} FN, {r.FP} FP"
                                    for r in at_sizes.itertuples())
        reports.at[image_id,'bukti_eksperimen'] = (
            f"{resolution_text}. Pada 640px dan ambang 0,25, "
            f"diperoleh {low['TP']} TP, {low['FN']} FN dan {low['FP']} FP."
        )
        thresholds_help = low['TP']>int(row.TP)
        resolution_changes = at_sizes.TP.nunique()>1 or at_sizes.FN.nunique()>1
        reports.at[image_id,'hipotesis_mekanisme'] = (
            'Sebagian mask yang cocok menghasilkan skor di bawah ambang 0,50.'
            if thresholds_help else
            'Grid representasi mask dan detail fitur bisa berubah bersama ukuran input.'
            if resolution_changes else
            'Kandidat kelas yang sama atau tumpang tindih mask perlu diperiksa pada overlay.'
        )
        reports.at[image_id,'kesimpulan_dan_batasan'] = (
            'Menurunkan ambang menambah deteksi yang cocok pada prediksi yang sama. Perubahan FP dicatat.'
            if thresholds_help else
            'Jumlah deteksi berubah ketika ukuran input diganti. Uji ini mengubah fitur dan grid secara bersamaan.'
            if resolution_changes else
            'Jumlah TP/FN tetap pada tiga ukuran dan ambang 0,25 tidak menambah TP. Penyebab spesifik belum terisolasi.'
        )
    reports.reset_index().to_csv(FAIL_CSV,index=False)
    pd.DataFrame(diagnostics,columns=['source_photo','image_id','gt_annotation_id','class','event',
        'best_IoU_same_class','max_confidence_same_class_IoU50']).to_csv(METRICS/'diagnostik_instance_HP.csv',index=False)
    display(reports[['source_photo','apa_yang_gagal','bukti_eksperimen','kesimpulan_dan_batasan']])


## Ringkasan angka hasil


In [ ]:
# Ringkasan numerik untuk log Kaggle dan pembacaan ulang hasil.
assert len(comp[['model_name','test_set']].drop_duplicates()) == 6
assert len(nrp_df)==9 and all(set(g.input_size)==set(RESOLUTIONS) for _,g in nrp_df.groupby('model_name'))
def clean_report(frame):
    return json.loads(frame.to_json(orient='records',double_precision=8))
result_columns = ['model_name','test_set','scope','n_images','n_categories_with_gt',
                  'mask_AP50_95','mask_AP50','mask_AP75','mask_AR100']
report = {
    'models':clean_report(comp[result_columns]),
    'revision_impact':clean_report(impact),
    'nrp':clean_report(nrp_df[['model_name','input_size','epochs_reached','training_seconds','mask_AP50','mask_AP50_95']]),
    'nrp_matched_epochs':clean_report(epoch_nrp),
    'hypotheses':clean_report(hypothesis_observations),
    'training':clean_report(pd.read_csv(TIME_LOG)[['model','run','seconds','status','gpu']]),
    'classes_common_original_hp':[CAT_NAMES[c] for c in COMMON_CAT_IDS],
    'hp_masks':sum(hp_freq.values()),
    'hp_images':len(HP_IDS),
    'hp_ground_truth_human_reviewed':HP_LABELS_REVIEWED,
    'source_v5_verified':SOURCE_V5_CONFIRMED,
    'failure_cases':len(FAIL_IDS),
}
write_json(METRICS/'ringkasan_hasil.json',report)
print('CPMK3_EVALUATION_JSON '+json.dumps(report,ensure_ascii=False,separators=(',',':')))


## 13. Simpan hasil dan bukti

Informasi EXIF diambil dari foto asli di ZIP atau folder. Isi catatan kondisi sulit dan kepemilikan foto sesuai sumbernya. Path folder tangkapan layar bisa diatur di bagian konfigurasi.

Checklist mencatat hasil yang sudah tersedia dan bukti yang masih perlu dicek. Setelah Save Version, buka Output Kaggle untuk mengambil ZIP hasil dan bukti serta berkas dari sesi pelatihan.

In [ ]:
def exif_record(name,raw):
    with Image.open(io.BytesIO(raw)) as pic:
        ex=pic.getexif();detail=ex.get_ifd(0x8769) if 0x8769 in ex else {}
        return {'source_photo':Path(name).name,'sha256':hashlib.sha256(raw).hexdigest(),
            'width':pic.width,'height':pic.height,'Make':str(ex.get(271,'')),'Model':str(ex.get(272,'')),
            'DateTimeOriginal':str(detail.get(36867,'')), 'ISO':str(detail.get(34855,'')),
            'ExposureTime':str(detail.get(33434,'')), 'Orientation':str(ex.get(274,'')),
            'camera_and_time':bool(ex.get(272) and detail.get(36867))}
# Cari foto asli berdasarkan manifest nama dan EXIF, bukan membaca metadata dari JPEG Roboflow.
exif_rows={}
photo_search=Path(ORIGINAL_PHOTOS_DIR) if ORIGINAL_PHOTOS_DIR else INPUT_ROOT
if photo_search.is_dir():
    for f in photo_search.rglob('*'):
        if f.is_file() and f.name in HP_SOURCE_NAMES and not ignored(f):
            row=exif_record(f.name,f.read_bytes())
            if row['camera_and_time'] or f.name not in exif_rows:exif_rows[f.name]=row
    for archive in sorted(photo_search.rglob('*.zip')):
        if ignored(archive):continue
        with zipfile.ZipFile(archive) as z:
            members=[n for n in z.namelist() if Path(n).name in HP_SOURCE_NAMES and not ignored(n)]
            for member in members:
                row=exif_record(member,z.read(member));name=row['source_photo']
                if row['camera_and_time'] or name not in exif_rows:exif_rows[name]=row
exif_df=pd.DataFrame(list(exif_rows.values()))
if not exif_df.empty:exif_df.to_csv(PROOF/'exif_40_foto_asli.csv',index=False)
exif_count=sum(row['camera_and_time'] for row in exif_rows.values())
manual=pd.read_csv(review_path,keep_default_na=False)
yes={'YA','YES','TRUE','SUDAH','OK'}
review_count=sum(str(x).upper() in yes for x in manual.reviewed_all_objects)
own_count=sum(str(x).upper() in yes for x in manual.own_phone_confirmed)
# Jenis kondisi perlu ditulis manusia dengan alasan; tidak ditebak hanya dari nama file.
hard_count=sum(bool(str(x).strip()) and str(x).upper() not in {'NORMAL','BIASA','TIDAK','NONE'} for x in manual.difficult_condition)
screens=[]
if SCREENSHOTS_DIR and Path(SCREENSHOTS_DIR).is_dir():
    screens=[p for p in Path(SCREENSHOTS_DIR).rglob('*') if p.suffix.lower() in ('.png','.jpg','.jpeg')]
write_json(PROOF/'evidence_status.json',{'exif_camera_time_count':exif_count,'human_review_count':review_count,
    'own_phone_confirmed_count':own_count,'hard_conditions_documented_count':hard_count,
    'screenshots_found':len(screens),'source_v5_confirmed':SOURCE_V5_CONFIRMED,
    'warning':'File/keterangan ada bukan bukti kebenaran isi; review manusia dan provenance tetap dibutuhkan.'})
# Template laporan tidak memuat hasil palsu dan tidak ditimpa saat run ulang.
reflection=PROOF/'REFLEKSI_DAN_AI.txt'
if not reflection.exists():
    reflection.write_text('''REFLEKSI DAN DEKLARASI AI - lengkapi sesuai proses nyata
Nama/NRP:
Asal Panoptic Indoor Segmentation v5 atau persetujuan dosen (URL/screenshot):
Kelengkapan label HP setelah revisi dan potensi objek kecil yang terlewat:
Tautan/nama screenshot anotasi Roboflow:
Minimal 10 kondisi sulit dan alasannya (lihat review_manual_40_hp.csv):
Hasil 3 model pada 2 test set (isi setelah run):
Hasil NRP 0 pada 320, 640, 960; jumlah epoch dan waktu aktual:
Analisis 5 kegagalan; uji hipotesis; batasan klaim:
Kendala dan keputusan nyata:
Alat AI yang dipakai dalam revisi ini adalah Codex dan Mask2Former COCO Panoptic. Kandidat mask dipilih melalui pemeriksaan visual dan ditambah poligon dari foto asli. Draf revisi diperiksa sebelum memeriksa prediksi model tugas pada test HP.
Prompt revisi yang diberikan: kenapa hasilnya gada ya, aku mau hasilnya ada, tapi aku rasa akurasinya jeelek karna anotasi ku salah mungkin, tolong benerin
Contoh saran AI yang perlu diperbaiki: rancangan awal NRP hanya menguji tiga ukuran di YOLO11n. Rubrik baru mewajibkan tabel dan grafik untuk ketiga model. Ditambah empat pelatihan dengan data dan recipe model yang sama.
Bukti perbaikannya: file validasi notebook dan hasil eksekusi yang benar-benar tersedia; jangan mengklaim GPU training telah diuji bila belum.
''',encoding='utf-8')
checks=[
 ('HP hanya di test dan berjumlah 40',len(HP_IDS)==40),('Test asli terpisah 359',len(OFFICIAL_IDS)==359),
 ('EXIF model kamera dan waktu pada 40 foto',exif_count>=40),
 ('Asal data latih v5 dibuktikan',SOURCE_V5_CONFIRMED and bool(SOURCE_V5_NOTE.strip())),
 ('Seluruh HP direview manusia',HP_LABELS_REVIEWED and review_count>=40),
 ('Foto HP sendiri dikonfirmasi',own_count>=40),('>=10 kondisi sulit didokumentasikan',hard_count>=10),
 ('Screenshot proses anotasi tersedia',bool(screens)),
 ('3 checkpoint baseline selesai',all(ck.is_file() for _,_,ck in BASELINES)),
 ('9 kombinasi model dan ukuran NRP dievaluasi',len(nrp_df)==9),
 ('3 model x 2 test set dievaluasi',not comp.empty and len(comp[['model_name','test_set']].drop_duplicates())==6),
 ('5 foto kegagalan nyata tersedia',len(FAIL_IDS)==5),
 ('Eksperimen kecil tersedia',(METRICS/'eksperimen_kecil_per_foto.csv').is_file()),
 ('Waktu training dicatat',TIME_LOG.is_file())]
status=pd.DataFrame(checks,columns=['requirement','verified_automatically_or_declared'])
status.to_csv(PROOF/'checklist_pengumpulan.csv',index=False);display(status)
if TIME_LOG.exists():display(pd.read_csv(TIME_LOG))
for name in ['YOLO11n','YOLO11s','MaskRCNN']:print(name,'training wall-time tercatat:',round(used_seconds(name)/60,2),'menit')
print('Jumlah GPU benar-benar sibuk tidak diukur profiler; durasi di log adalah wall-clock training pada device GPU.')
print('Semua hasil:',WORK)
# Hasil dapat diunduh dari panel Output Kaggle. Gambar dataset turunan tidak disalin ke arsip hasil.
result_archive=WORK_ROOT/('hasil_cpmk3_'+DATA_FINGERPRINT[:12]+'.zip')
with zipfile.ZipFile(result_archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for folder in (PROOF,METRICS):
        for f in folder.rglob('*'):
            if f.is_file():z.write(f,f.relative_to(WORK))
    for name,run_dir in sorted(run_locations.items()):
        for filename in ('run_manifest.json','complete.json','heartbeat.json','args.yaml','results.csv','maskrcnn_train.csv'):
            f=run_dir/filename
            if f.is_file():z.write(f,str(Path('runs')/name/filename))
print('Arsip hasil/bukti/checkpoint:',result_archive)


## Coba model dengan foto baru


In [ ]:
# Ubah tiga nilai ini jika memakai file input Kaggle saat sesi demo.
DEMO_PATH = None
DEMO_MODEL = 'YOLO11s'
DEMO_THRESHOLD = 0.35

def predict_demo(photo, model_name):
    '''Inferensi foto baru menggunakan checkpoint 640.'''
    if isinstance(photo,(bytes,bytearray,memoryview)):
        photo = io.BytesIO(bytes(photo))
    with Image.open(photo) as img:
        image = ImageOps.exif_transpose(img).convert('RGB')
        image.thumbnail((1920,1920))
        arr = np.asarray(image).copy()
    options = {name:(kind,ckpt) for name,kind,ckpt in BASELINES}
    kind,weights = options[model_name]
    predictor = load_run_model(kind,weights,BASE_SIZE)
    try:
        pred = model_predict_image(predictor,kind,arr,BASE_SIZE)
    finally:
        del predictor
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    return arr,pred

def render_demo(arr, predictions, threshold, model_name):
    '''Ubah ambang skor dan gambar ulang mask dari prediksi foto yang sama.'''
    if not PRED_CONF <= threshold <= 1.0:
        raise ValueError(f'Gunakan ambang antara {PRED_CONF} dan 1.')
    overlay = arr.copy()
    h,w = arr.shape[:2]
    kept = [p for p in predictions if p['score']>=threshold]
    for index,p in enumerate(sorted(kept,key=lambda row:row['score'])):
        mask = decode_seg(p['segmentation'],h,w).astype(bool)
        if not mask.any():
            continue
        color = np.random.default_rng(index+41).integers(65,245,3)
        overlay[mask] = (0.56*overlay[mask] + 0.44*color).astype(np.uint8)
        contour,_ = cv2.findContours(mask.astype(np.uint8),cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(overlay,contour,-1,tuple(int(v) for v in color),2)
        ys,xs = np.where(mask)
        label = f"{CAT_NAMES[p['category_id']]} {p['score']:.2f}"
        point = int(xs.min()),max(18,int(ys.min()))
        cv2.putText(overlay,label,point,cv2.FONT_HERSHEY_SIMPLEX,0.65,(0,0,0),4,cv2.LINE_AA)
        cv2.putText(overlay,label,point,cv2.FONT_HERSHEY_SIMPLEX,0.65,(255,255,255),1,cv2.LINE_AA)
    fig,axes = plt.subplots(1,2,figsize=(11,6))
    for ax,graphic,title in zip(axes,(arr,overlay),('Foto baru','Mask hasil inferensi')):
        ax.imshow(graphic)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{model_name} · ambang {threshold:.2f} · {len(kept)} mask')
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    if kept:
        print(pd.DataFrame([{'kelas':CAT_NAMES[c],'jumlah':v} for c,v in Counter(p['category_id'] for p in kept).items()]).to_string(index=False))

def ap_satu_kelas(model_name, category, split='original'):
    '''Tampilkan AP mask per kelas dari evaluasi tersimpan.'''
    key,scope = (f'{model_name}_valid','kelas_ada_di_train') if split=='validation' else (
        f'{model_name}_{split}_test','semua_kelas')
    file = METRICS/key/f'AP_per_class_{scope}.csv'
    if not file.is_file():
        print('Tabel AP belum tersedia untuk pilihan ini.')
        return None
    data = pd.read_csv(file)
    row = data.loc[data['class']==category]
    if row.empty or pd.isna(row.iloc[0]['mask_AP50_95']):
        print(f'{category} tidak memiliki ground truth pada set ini.')
        return None
    result = float(row.iloc[0]['mask_AP50_95'])*100
    print(f'{model_name} · {category} · {split} · mask AP50–95 {result:.2f}%')
    return result

def run_live_demo(photo, model_name=DEMO_MODEL, threshold=DEMO_THRESHOLD):
    arr,predictions = predict_demo(photo,model_name)
    render_demo(arr,predictions,threshold,model_name)

if DEMO_PATH is not None:
    run_live_demo(DEMO_PATH, DEMO_MODEL, DEMO_THRESHOLD)
else:
    try:
        import ipywidgets as widgets
        from IPython.display import clear_output
    except ImportError:
        print('Isi DEMO_PATH dengan path foto baru, lalu jalankan ulang cell ini.')
    else:
        uploader = widgets.FileUpload(accept='image/jpeg,image/png',multiple=False,description='Pilih foto')
        choices = [(name,name) for name,_,_ in BASELINES]
        demo_pick = widgets.Dropdown(options=choices,value=DEMO_MODEL,description='Model')
        cutoff = widgets.FloatSlider(value=DEMO_THRESHOLD,min=0.01,max=0.95,step=0.01,
                                     description='Ambang',continuous_update=False)
        execute_demo = widgets.Button(description='Jalankan inferensi',button_style='primary')
        canvas = widgets.Output()
        demo_state = {}

        def redraw_demo(_=None):
            if not demo_state:
                return
            with canvas:
                clear_output(wait=True)
                render_demo(demo_state['photo'],demo_state['predictions'],cutoff.value,demo_state['model'])

        def infer_uploaded(_):
            with canvas:
                clear_output(wait=True)
                items = list(uploader.value.values()) if isinstance(uploader.value,dict) else list(uploader.value)
                if not items:
                    print('Pilih satu foto JPG atau PNG yang baru diambil.')
                    return
                try:
                    arr,pred = predict_demo(items[0]['content'],demo_pick.value)
                except (ValueError,OSError,RuntimeError) as error:
                    print(f'Inferensi gagal. {error}')
                    return
                demo_state.update(photo=arr,predictions=pred,model=demo_pick.value)
            redraw_demo()

        ap_class = widgets.Dropdown(options=sorted(CAT_NAMES[c] for c in CAT_IDS),value='chair',description='Kelas')
        ap_split = widgets.Dropdown(options=[('Test asli','original'),('Test HP','hp'),('Validation','validation')],
                                    value='original',description='Data uji')
        ap_button = widgets.Button(description='Tampilkan AP')
        ap_output = widgets.Output()

        def report_ap(_):
            with ap_output:
                clear_output(wait=True)
                ap_satu_kelas(demo_pick.value,ap_class.value,ap_split.value)

        cutoff.observe(redraw_demo,names='value')
        execute_demo.on_click(infer_uploaded)
        ap_button.on_click(report_ap)
        display(widgets.VBox([uploader,demo_pick,cutoff,execute_demo,canvas,ap_class,ap_split,ap_button,ap_output]))
        print('Pada sesi aktif, unggah foto baru dan klik Jalankan inferensi. '
              'Gunakan Ambang untuk mengubah mask yang ditampilkan, atau Tampilkan AP untuk melihat satu kelas.')


## Kalau muncul error

| Yang terlihat | Cara mengeceknya |
|---|---|
| Dataset yang sesuai belum ketemu | Tambahkan ekspor v2 final-test40 yang berisi JSON COCO dan JPG sebagai Input. Daftar path bisa dicek di sel 3. |
| Ada beberapa ekspor | Isi `DATASET_OVERRIDE` dengan folder latih, validasi, dan uji, atau ZIP ekspor v2 yang sesuai. |
| Asal data dari v5 belum dikonfirmasi | Cek riwayat impor atau fork di Roboflow, atau minta konfirmasi dosen. Gunakan bukti yang menunjukkan versi sumbernya. Audit bisa dijalankan sambil menunggu. |
| Mask, RLE, atau ID bermasalah | Periksa anotasi di proyek lalu ekspor ulang ke COCO. Pertahankan mask yang sesuai dengan objek saat memperbaiki data. |
| CUDA out of memory | Turunkan `BATCH_YOLO` dari 4 ke 2 untuk semua ukuran NRP. Pakai `WORK_ROOT` baru supaya hasil konfigurasi ini tersimpan sendiri. Pertahankan ukuran eksperimen. Batch Mask R-CNN sudah 1. |
| NMS dari torch atau torchvision gagal | Restart sesi GPU standar Kaggle. Pakai pasangan torch dan torchvision yang sesuai dengan CUDA. |
| Internet atau bobot gagal diunduh | Aktifkan internet di Kaggle dan jalankan sel instalasi lagi. Kalau perlu offline, tambahkan pustaka dan bobot resmi sebagai Input, lalu sesuaikan path. |
| Checkpoint atau skor belum muncul | Cek `RUN_TRAINING`, log error, dan apakah pelatihan sempat menyelesaikan epoch yang dinilai di validasi. Untuk evaluasi, cek juga apakah output pelatihan sudah ditambahkan sebagai Input. |
| Sesi berhenti di tengah jalan | Simpan output, log progres, dan waktu yang terpakai. Kalau perlu mulai lagi, gunakan run ID baru. |
| Label HP perlu diperbaiki setelah pelatihan | Tulis alasan revisi berdasarkan foto dan kelas acuan. Catat perubahan label sebelum menghitung ulang skor. Fingerprint baru membantu mencocokkan hasil dengan versi label yang dipakai. |

## Sumber yang dipakai

- Instruksi tugas dari PDF, slide 29–32.
- [Panoptic Indoor Segmentation v5 di Roboflow Universe](https://universe.roboflow.com/panopticindoor/panoptic-indoor-segmentation/dataset/5)
- [Bukti anotasi Roboflow di laporan](../laporan/Visi_Komputer_Laporan_Chelsea_5054241010_Revisi.pdf#page=2)
- [Panduan pelatihan YOLO](https://docs.ultralytics.com/modes/train/) dan [callback Ultralytics](https://docs.ultralytics.com/usage/callbacks/)
- [Ultralytics versi 8.3.228](https://pypi.org/project/ultralytics/8.3.228/)
- [Mask R-CNN di torchvision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn.html)
- [COCOeval](https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py)
- [Panduan notebook Kaggle](https://www.kaggle.com/docs/notebooks)
